Generate a Tactile Stimulus for Woojer, according to specs from:
Lerner, F., Tahar, G., Bar, A., Koren, O., & Flash, T. (2021). VR Setup to Assess Peripersonal Space Audio-Tactile 3D Boundaries. Frontiers in virtual reality, 2, 644214.

In [3]:
"""
========================================================================
TACTILE AUDIO STIMULUS GENERATOR
Based on Lerner et al. (2021) Frontiers in Virtual Reality
========================================================================

Generates tactile audio stimuli for PPS (Peripersonal Space) experiments.

Tactile stimulus specifications from Lerner et al.:
- Attack signal: 200 Hz sinusoidal wave, amplitude +4 dB
- Decay signal: 50 Hz, amplitude -22 dB
- Duration: 100 ms
- These are delivered to haptic devices (e.g., Woojer strap)

This script generates the audio file that will drive the tactile actuator.
========================================================================
"""

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import os

print("=" * 70)
print("TACTILE AUDIO STIMULUS GENERATOR")
print("Based on Lerner et al. (2021)")
print("=" * 70)

# ========================================================================
# PARAMETERS
# ========================================================================

# Audio parameters
SAMPLE_RATE = 44100         # Sample rate in Hz
DURATION = 0.1              # Duration in seconds (100 ms)

# Signal parameters from Lerner et al.
ATTACK_FREQ = 200           # Attack signal frequency (Hz)
ATTACK_DB = 4               # Attack signal amplitude (dB)

DECAY_FREQ = 50             # Decay signal frequency (Hz)
DECAY_DB = -22              # Decay signal amplitude (dB)

# Output directory
OUTPUT_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\TactileStimuli"

# ========================================================================
# HELPER FUNCTIONS
# ========================================================================

def db_to_amplitude(db):
    """Convert dB to linear amplitude (relative to reference of 1.0)"""
    return 10 ** (db / 20)

def generate_sinusoid(frequency, duration, sample_rate, amplitude=1.0):
    """Generate a sinusoidal wave"""
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    signal = amplitude * np.sin(2 * np.pi * frequency * t)
    return signal, t

def apply_envelope(signal, attack_time=0.005, release_time=0.01):
    """
    Apply attack and release envelope to avoid clicks
    
    Parameters:
    - attack_time: fade-in duration in seconds (5 ms default)
    - release_time: fade-out duration in seconds (10 ms default)
    """
    num_samples = len(signal)
    envelope = np.ones(num_samples)
    
    # Attack (fade in)
    attack_samples = int(attack_time * SAMPLE_RATE)
    if attack_samples > 0:
        envelope[:attack_samples] = np.linspace(0, 1, attack_samples)
    
    # Release (fade out)
    release_samples = int(release_time * SAMPLE_RATE)
    if release_samples > 0:
        envelope[-release_samples:] = np.linspace(1, 0, release_samples)
    
    return signal * envelope

# ========================================================================
# GENERATE TACTILE STIMULUS
# ========================================================================

print("\nGenerating tactile stimulus...")
print("-" * 70)

# Convert dB to amplitude
attack_amplitude = db_to_amplitude(ATTACK_DB)
decay_amplitude = db_to_amplitude(DECAY_DB)

print(f"Attack signal: {ATTACK_FREQ} Hz at {ATTACK_DB} dB (amplitude: {attack_amplitude:.4f})")
print(f"Decay signal:  {DECAY_FREQ} Hz at {DECAY_DB} dB (amplitude: {decay_amplitude:.4f})")
print(f"Duration:      {DURATION * 1000:.0f} ms")
print(f"Sample rate:   {SAMPLE_RATE} Hz")

# Generate attack signal (200 Hz)
attack_signal, time = generate_sinusoid(
    ATTACK_FREQ, 
    DURATION, 
    SAMPLE_RATE, 
    attack_amplitude
)

# Generate decay signal (50 Hz)
decay_signal, _ = generate_sinusoid(
    DECAY_FREQ, 
    DURATION, 
    SAMPLE_RATE, 
    decay_amplitude
)

# Combine signals (sum of attack and decay)
tactile_signal = attack_signal + decay_signal

# Apply envelope to avoid clicks at start/end
tactile_signal = apply_envelope(tactile_signal)

# Normalize to prevent clipping
max_amplitude = np.max(np.abs(tactile_signal))
if max_amplitude > 0:
    tactile_signal = tactile_signal / max_amplitude * 0.95

print(f"\n✓ Signal generated ({len(tactile_signal)} samples)")

# ========================================================================
# SAVE AUDIO FILE
# ========================================================================

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save as WAV file
output_filename = "tactile_stimulus_0ms.wav"
output_path = os.path.join(OUTPUT_DIR, output_filename)

sf.write(output_path, tactile_signal, SAMPLE_RATE)

print(f"\n✓ Saved: {output_filename}")
print(f"   Location: {OUTPUT_DIR}")

# ========================================================================
# VISUALIZATION
# ========================================================================

print("\nCreating visualization...")

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Time axis in milliseconds
time_ms = time * 1000

# Plot 1: Complete tactile signal
axes[0].plot(time_ms, tactile_signal, 'b-', linewidth=1.5)
axes[0].set_title('Tactile Stimulus (Attack + Decay)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time (ms)', fontsize=11)
axes[0].set_ylabel('Amplitude', fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

# Plot 2: Individual components
axes[1].plot(time_ms, attack_signal, 'r-', linewidth=1, label=f'Attack ({ATTACK_FREQ} Hz, +{ATTACK_DB} dB)', alpha=0.7)
axes[1].plot(time_ms, decay_signal, 'g-', linewidth=1, label=f'Decay ({DECAY_FREQ} Hz, {DECAY_DB} dB)', alpha=0.7)
axes[1].set_title('Signal Components', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time (ms)', fontsize=11)
axes[1].set_ylabel('Amplitude', fontsize=11)
axes[1].legend(fontsize=10, loc='upper right')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

# Plot 3: Frequency spectrum
from scipy import signal as sp_signal
freqs, psd = sp_signal.welch(tactile_signal, SAMPLE_RATE, nperseg=len(tactile_signal))

axes[2].semilogy(freqs, psd, 'b-', linewidth=1.5)
axes[2].set_title('Frequency Spectrum', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Frequency (Hz)', fontsize=11)
axes[2].set_ylabel('Power Spectral Density', fontsize=11)
axes[2].grid(True, alpha=0.3, which='both')
axes[2].set_xlim(0, 500)  # Focus on relevant frequency range

# Mark the main frequencies
axes[2].axvline(x=ATTACK_FREQ, color='r', linestyle='--', alpha=0.7, label=f'{ATTACK_FREQ} Hz (Attack)')
axes[2].axvline(x=DECAY_FREQ, color='g', linestyle='--', alpha=0.7, label=f'{DECAY_FREQ} Hz (Decay)')
axes[2].legend(fontsize=10)

plt.tight_layout()

# Save figure
plot_filename = "tactile_stimulus_0ms_visualization.png"
plot_path = os.path.join(OUTPUT_DIR, plot_filename)
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"✓ Saved visualization: {plot_filename}")

# ========================================================================
# SUMMARY
# ========================================================================

print("\n" + "=" * 70)
print("GENERATION COMPLETE")
print("=" * 70)

print(f"\nGenerated tactile stimulus:")
print(f"  • Filename:   {output_filename}")
print(f"  • Duration:   {DURATION * 1000:.0f} ms")
print(f"  • Sample rate: {SAMPLE_RATE} Hz")
print(f"  • Attack:     {ATTACK_FREQ} Hz at {ATTACK_DB} dB")
print(f"  • Decay:      {DECAY_FREQ} Hz at {DECAY_DB} dB")

print(f"\nFiles saved to:")
print(f"  {OUTPUT_DIR}")

print(f"\nUsage:")
print(f"  • This audio file drives haptic actuators (e.g., Woojer strap)")
print(f"  • Connect to haptic device via audio interface")
print(f"  • Calibrate intensity based on participant threshold")
print(f"  • Use in audio-tactile PPS experiments")

print("\n" + "=" * 70)

TACTILE AUDIO STIMULUS GENERATOR
Based on Lerner et al. (2021)

Generating tactile stimulus...
----------------------------------------------------------------------
Attack signal: 200 Hz at 4 dB (amplitude: 1.5849)
Decay signal:  50 Hz at -22 dB (amplitude: 0.0794)
Duration:      100 ms
Sample rate:   44100 Hz

✓ Signal generated (4410 samples)

✓ Saved: tactile_stimulus_0ms.wav
   Location: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\TactileStimuli

Creating visualization...
✓ Saved visualization: tactile_stimulus_0ms_visualization.png

GENERATION COMPLETE

Generated tactile stimulus:
  • Filename:   tactile_stimulus_0ms.wav
  • Duration:   100 ms
  • Sample rate: 44100 Hz
  • Attack:     200 Hz at 4 dB
  • Decay:      50 Hz at -22 dB

Files saved to:
  G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\TactileStimuli

Usage:
  • This audio file drives haptic actuators (e.g., Woojer strap)
  • Connect to haptic device via audio interface
  • Calibra

Add Delays (D1 - D5) to tactile

In [4]:
"""
# Tactile Stimulus Onset Delay Generator

**Purpose:** Create tactile stimuli with precise onset delays for PPS experiments

## Overview

This script takes a base tactile stimulus (100ms vibration) and creates 5 versions,
each with a specific duration of silence prepended. These delayed versions allow for
synchronized audio-tactile presentation in PPS boundary experiments.

## Stimulus Onset Asynchrony (SOA) Delays

The following delays correspond to different perceived distances in PPS paradigm:

| Delay | Time from auditory onset | Perceived Distance |
|-------|--------------------------|-------------------|
| T1    | 300 ms                   | Far               |
| T2    | 800 ms                   | Medium-Far        |
| T3    | 1500 ms                  | Medium            |
| T4    | 2200 ms                  | Medium-Near       |
| T5    | 2700 ms                  | Near              |

## Method

1. Load base tactile stimulus (tactile_stimulus_0ms.wav)
2. For each delay: prepend specified silence duration
3. Save as new file with delay in filename
4. Total duration = delay + tactile_duration (delay + 100ms)

## Output Files

- tactile_stimulus_300ms.wav  (300ms silence + 100ms tactile = 400ms total)
- tactile_stimulus_800ms.wav  (800ms silence + 100ms tactile = 900ms total)
- tactile_stimulus_1500ms.wav (1500ms silence + 100ms tactile = 1600ms total)
- tactile_stimulus_2200ms.wav (2200ms silence + 100ms tactile = 2300ms total)
- tactile_stimulus_2700ms.wav (2700ms silence + 100ms tactile = 2800ms total)

## Usage in Experiments

These files can be played simultaneously with looming auditory stimuli (3000ms duration).
The tactile sensation will occur at precise timepoints during the auditory approach,
allowing measurement of RT as a function of perceived sound source distance.

========================================================================
"""

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import os

print("=" * 70)
print("TACTILE STIMULUS ONSET DELAY GENERATOR")
print("=" * 70)

# ========================================================================
# PARAMETERS
# ========================================================================

# Base tactile stimulus (no delay)
BASE_STIMULUS_PATH = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\TactileStimuli\tactile_stimulus_0ms.wav"

# Output directory
OUTPUT_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\TactileStimuli"

# Onset delays in milliseconds (SOA - Stimulus Onset Asynchrony)
ONSET_DELAYS_MS = [300, 800, 1500, 2200, 2700]

# ========================================================================
# LOAD BASE STIMULUS
# ========================================================================

print("\nLoading base tactile stimulus...")
print("-" * 70)

try:
    # Load the base stimulus
    base_stimulus, sample_rate = sf.read(BASE_STIMULUS_PATH)
    
    print(f"✓ Loaded: {os.path.basename(BASE_STIMULUS_PATH)}")
    print(f"  Sample rate: {sample_rate} Hz")
    print(f"  Duration: {len(base_stimulus) / sample_rate * 1000:.1f} ms")
    print(f"  Samples: {len(base_stimulus)}")
    
    # If stereo, convert to mono
    if len(base_stimulus.shape) > 1:
        base_stimulus = base_stimulus[:, 0]
        print(f"  Converted stereo to mono")
    
except FileNotFoundError:
    print(f"✗ ERROR: Base stimulus file not found!")
    print(f"  Expected location: {BASE_STIMULUS_PATH}")
    print(f"\nPlease ensure the base tactile stimulus exists.")
    print(f"You may need to run the tactile stimulus generator first.")
    exit(1)

except Exception as e:
    print(f"✗ ERROR loading base stimulus: {e}")
    exit(1)

# ========================================================================
# GENERATE DELAYED STIMULI
# ========================================================================

print("\n" + "=" * 70)
print("GENERATING DELAYED TACTILE STIMULI")
print("=" * 70)

# Create output directory if needed
os.makedirs(OUTPUT_DIR, exist_ok=True)

generated_files = []
visualizations = []

for i, delay_ms in enumerate(ONSET_DELAYS_MS, start=1):
    print(f"\n[{i}/{len(ONSET_DELAYS_MS)}] Creating stimulus with {delay_ms} ms delay...")
    
    # Calculate number of silence samples needed
    delay_seconds = delay_ms / 1000.0
    silence_samples = int(delay_seconds * sample_rate)
    
    # Create silence array
    silence = np.zeros(silence_samples)
    
    # Concatenate silence + base stimulus
    delayed_stimulus = np.concatenate([silence, base_stimulus])
    
    # Calculate total duration
    total_duration_ms = len(delayed_stimulus) / sample_rate * 1000
    
    # Create output filename
    output_filename = f"tactile_stimulus_{delay_ms}ms.wav"
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    
    # Save the delayed stimulus
    sf.write(output_path, delayed_stimulus, sample_rate)
    
    print(f"  ✓ Saved: {output_filename}")
    print(f"    Delay: {delay_ms} ms ({silence_samples} samples)")
    print(f"    Tactile: {len(base_stimulus) / sample_rate * 1000:.1f} ms")
    print(f"    Total duration: {total_duration_ms:.1f} ms")
    
    generated_files.append(output_filename)
    visualizations.append({
        'delay_ms': delay_ms,
        'signal': delayed_stimulus,
        'silence_samples': silence_samples
    })

# ========================================================================
# CREATE VISUALIZATION
# ========================================================================

print("\n" + "-" * 70)
print("Creating visualization...")

fig, axes = plt.subplots(len(ONSET_DELAYS_MS), 1, figsize=(14, 10), sharex=True)

for idx, viz_data in enumerate(visualizations):
    delay_ms = viz_data['delay_ms']
    signal = viz_data['signal']
    silence_samples = viz_data['silence_samples']
    
    # Time axis in milliseconds
    time_ms = np.linspace(0, len(signal) / sample_rate * 1000, len(signal))
    
    # Plot signal
    axes[idx].plot(time_ms, signal, 'b-', linewidth=1)
    axes[idx].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
    
    # Highlight the silence period
    axes[idx].axvspan(0, delay_ms, alpha=0.2, color='gray', label='Silence')
    
    # Highlight the tactile period
    tactile_start = delay_ms
    tactile_end = time_ms[-1]
    axes[idx].axvspan(tactile_start, tactile_end, alpha=0.2, color='red', label='Tactile')
    
    # Mark the onset
    axes[idx].axvline(x=delay_ms, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    # Labels
    axes[idx].set_ylabel(f'T{idx+1}\n({delay_ms}ms)', fontsize=10, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim(-1.1, 1.1)
    
    # Add text annotation
    axes[idx].text(delay_ms / 2, 0.8, f'{delay_ms} ms delay', 
                   ha='center', fontsize=9, color='gray', fontweight='bold')
    
    if idx == 0:
        axes[idx].legend(loc='upper right', fontsize=9)
        axes[idx].set_title('Tactile Stimuli with Onset Delays (SOA)', 
                           fontsize=14, fontweight='bold', pad=10)

axes[-1].set_xlabel('Time (ms)', fontsize=12, fontweight='bold')
axes[len(ONSET_DELAYS_MS)//2].set_ylabel('Amplitude', fontsize=12, fontweight='bold', 
                                          position=(0, 0.5))

plt.tight_layout()

# Save visualization
viz_filename = "tactile_stimuli_onset_delays_overview.png"
viz_path = os.path.join(OUTPUT_DIR, viz_filename)
plt.savefig(viz_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"✓ Saved visualization: {viz_filename}")

# ========================================================================
# CREATE TIMING DIAGRAM
# ========================================================================

print("Creating timing diagram...")

fig, ax = plt.subplots(figsize=(14, 8))

# Draw timeline for each stimulus
y_positions = np.arange(len(ONSET_DELAYS_MS), 0, -1)

for idx, viz_data in enumerate(visualizations):
    delay_ms = viz_data['delay_ms']
    y_pos = y_positions[idx]
    
    tactile_duration_ms = len(base_stimulus) / sample_rate * 1000
    total_duration = delay_ms + tactile_duration_ms
    
    # Draw silence period (gray)
    ax.barh(y_pos, delay_ms, left=0, height=0.6, 
            color='lightgray', edgecolor='black', linewidth=1.5, label='Silence' if idx == 0 else '')
    
    # Draw tactile period (red)
    ax.barh(y_pos, tactile_duration_ms, left=delay_ms, height=0.6, 
            color='lightcoral', edgecolor='black', linewidth=1.5, label='Tactile' if idx == 0 else '')
    
    # Add onset marker
    ax.plot([delay_ms, delay_ms], [y_pos - 0.35, y_pos + 0.35], 
            'r-', linewidth=3, zorder=10)
    
    # Add labels
    ax.text(-100, y_pos, f'T{idx+1}', fontsize=12, fontweight='bold', ha='right', va='center')
    ax.text(delay_ms, y_pos + 0.5, f'{delay_ms} ms', fontsize=9, ha='center', fontweight='bold')

# Reference line for 3000ms (looming sound duration)
ax.axvline(x=3000, color='blue', linestyle='--', linewidth=2, alpha=0.5, label='Looming sound end (3000ms)')

# Formatting
ax.set_xlabel('Time (ms)', fontsize=12, fontweight='bold')
ax.set_ylabel('Stimulus Onset Asynchrony (SOA)', fontsize=12, fontweight='bold')
ax.set_title('Tactile Stimulus Timing Diagram\n(Relative to Auditory Looming Onset)', 
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylim(0.5, len(ONSET_DELAYS_MS) + 0.5)
ax.set_xlim(-200, 3200)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, axis='x', alpha=0.3)
ax.set_yticks([])

plt.tight_layout()

# Save timing diagram
timing_filename = "tactile_stimuli_timing_diagram.png"
timing_path = os.path.join(OUTPUT_DIR, timing_filename)
plt.savefig(timing_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"✓ Saved timing diagram: {timing_filename}")

# ========================================================================
# SUMMARY
# ========================================================================

print("\n" + "=" * 70)
print("GENERATION COMPLETE")
print("=" * 70)

print(f"\nGenerated {len(generated_files)} delayed tactile stimuli:")
for i, filename in enumerate(generated_files, start=1):
    delay = ONSET_DELAYS_MS[i-1]
    print(f"  {i}. {filename:30s} (T{i}: {delay} ms delay)")

print(f"\nBase stimulus:")
print(f"  • {os.path.basename(BASE_STIMULUS_PATH)}")
print(f"  • Duration: {len(base_stimulus) / sample_rate * 1000:.1f} ms")

print(f"\nOutput location:")
print(f"  {OUTPUT_DIR}")

print(f"\nVisualizations created:")
print(f"  • {viz_filename}")
print(f"  • {timing_filename}")

print(f"\nExperimental usage:")
print(f"  • Play looming sound (3000 ms) and tactile stimulus simultaneously")
print(f"  • Tactile sensation occurs at T1-T5 timepoints during approach")
print(f"  • Measure participant RT to tactile stimulus at each SOA")
print(f"  • RT changes indicate PPS boundary location")

print("\n" + "=" * 70)

TACTILE STIMULUS ONSET DELAY GENERATOR

Loading base tactile stimulus...
----------------------------------------------------------------------
✓ Loaded: tactile_stimulus_0ms.wav
  Sample rate: 44100 Hz
  Duration: 100.0 ms
  Samples: 4410

GENERATING DELAYED TACTILE STIMULI

[1/5] Creating stimulus with 300 ms delay...
  ✓ Saved: tactile_stimulus_300ms.wav
    Delay: 300 ms (13230 samples)
    Tactile: 100.0 ms
    Total duration: 400.0 ms

[2/5] Creating stimulus with 800 ms delay...
  ✓ Saved: tactile_stimulus_800ms.wav
    Delay: 800 ms (35280 samples)
    Tactile: 100.0 ms
    Total duration: 900.0 ms

[3/5] Creating stimulus with 1500 ms delay...
  ✓ Saved: tactile_stimulus_1500ms.wav
    Delay: 1500 ms (66150 samples)
    Tactile: 100.0 ms
    Total duration: 1600.0 ms

[4/5] Creating stimulus with 2200 ms delay...
  ✓ Saved: tactile_stimulus_2200ms.wav
    Delay: 2200 ms (97020 samples)
    Tactile: 100.0 ms
    Total duration: 2300.0 ms

[5/5] Creating stimulus with 2700 ms de

Simplified Frontal Looming Sound Generator

In [1]:
"""
Simplified Frontal Looming Sound Generator

This script creates binaural audio files simulating a sound source approaching the listener
from directly in front (0° azimuth), using HRTF for spatial filtering and a physically-based
amplitude envelope for distance perception.

Key features:
- Pure frontal approach (no ITD, no ILD)
- Physically-based inverse square law for amplitude
- Multiple noise types supported
- Simple, transparent processing chain
"""
import numpy as np
import scipy.signal as signal
import soundfile as sf
import matplotlib.pyplot as plt
import os

# Check for required libraries
try:
    import sofar
    SOFA_AVAILABLE = True
except ImportError:
    print("Warning: sofar library not found. Please install with: pip install sofar --break-system-packages")
    SOFA_AVAILABLE = False

#############################################################
# CONTROL PARAMETERS
#############################################################

# Spatial parameters
INITIAL_DISTANCE = 2.0      # Starting distance in meters
FINAL_DISTANCE = 0.5        # Final distance in meters
AZIMUTH_ANGLE = 0           # Frontal approach (0 degrees)
ELEVATION_ANGLE = 0         # At ear level

# Audio parameters
DURATION = 3.0              # Duration of approach in seconds
SAMPLE_RATE = 44100         # Audio sample rate in Hz
VOLUME_LEVEL = 0.7          # Output volume (0.0 to 1.0)

# Distance attenuation using inverse square law
# Intensity = 1 / distance^2
# Sound pressure level (amplitude) = 1 / distance
USE_INVERSE_SQUARE_LAW = True

# Noise types to generate (will be numbered sequentially)
NOISE_TYPES = ['pink', 'blue', 'white', 'brown']

# SOFA file path
SOFA_FILE = r"C:\Users\cogpsy-vrlab\Downloads\HRIRs_neutral_head_orientation_v4\SOFA\FABIAN_HRIR_measured_HATO_0.sofa"

# Output directory
OUTPUT_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\LoomingStimuli"

#############################################################
# NOISE GENERATION
#############################################################

def generate_noise(noise_type, duration, fs):
    """Generate different types of noise with consistent spectral characteristics"""
    num_samples = int(duration * fs)
    
    # Use fixed seed for reproducibility
    np.random.seed(42)
    white_noise = np.random.randn(num_samples)
    
    if noise_type == 'white':
        noise = white_noise
        
    elif noise_type == 'pink':
        # Pink noise (1/f spectrum) using Voss-McCartney algorithm approximation
        b = np.array([0.049922035, -0.095993537, 0.050612699, -0.004408786])
        a = np.array([1.0, -2.494956002, 2.017265875, -0.522189400])
        noise = signal.lfilter(b, a, white_noise)
        
    elif noise_type == 'brown':
        # Brown/Brownian noise (1/f^2 spectrum)
        b = [1.0]
        a = [1.0, -0.98]
        noise = signal.lfilter(b, a, white_noise)
        
    elif noise_type == 'blue':
        # Blue noise (f spectrum, high-frequency emphasis)
        b = [1.0, -1.0]
        a = [1.0, -0.98]
        noise = signal.lfilter(b, a, white_noise)
        
    else:
        print(f"Warning: Noise type '{noise_type}' not recognized. Using white noise.")
        noise = white_noise
    
    # Normalize to prevent clipping
    noise = noise / np.max(np.abs(noise)) * 0.8
    
    return noise

#############################################################
# SOFA HRTF HANDLING
#############################################################

def load_sofa_file(sofa_path):
    """Load HRTF data from SOFA file"""
    try:
        print(f"Loading SOFA file: {os.path.basename(sofa_path)}")
        hrtf = sofar.read_sofa(sofa_path)
        
        print(f"SOFA Convention: {hrtf.GLOBAL_SOFAConventions} v{hrtf.GLOBAL_SOFAConventionsVersion}")
        
        # Get basic dimensions
        if hasattr(hrtf, 'Data_IR'):
            print(f"HRTF shape: {hrtf.Data_IR.shape}")
        
        return hrtf
        
    except Exception as e:
        print(f"Error loading SOFA file: {e}")
        import traceback
        traceback.print_exc()
        return None

def find_frontal_hrtf(hrtf, target_azimuth=0, target_elevation=0):
    """Find the HRTF measurement closest to frontal position (0°, 0°)"""
    try:
        # Check if this is a CTF (Common Transfer Function) file
        is_ctf = "CTF" in str(hrtf.GLOBAL_SOFAConventions)
        
        if is_ctf or (hasattr(hrtf, 'Data_IR') and hrtf.Data_IR.shape[0] <= 2 and hrtf.Data_IR.shape[1] == 1):
            print("Detected CTF file - using single filter")
            return 0, np.array([0.0, 0.0, 0.0])
        
        # For regular HRTF files, find closest to target position
        source_positions = np.array(hrtf.SourcePosition)
        
        # Handle different array shapes
        if source_positions.shape[0] == 3 and len(source_positions.shape) == 2:
            source_positions = source_positions.T
        
        # Determine if positions are in degrees or radians
        max_azimuth = np.max(np.abs(source_positions[:, 0]))
        positions_in_degrees = max_azimuth > 6.28
        
        if positions_in_degrees:
            az_target = target_azimuth
            el_target = target_elevation
        else:
            az_target = np.deg2rad(target_azimuth)
            el_target = np.deg2rad(target_elevation)
        
        # Find nearest position (prioritize azimuth match)
        distances = np.sqrt(
            (source_positions[:, 0] - az_target)**2 + 
            (source_positions[:, 1] - el_target)**2
        )
        
        nearest_idx = np.argmin(distances)
        nearest_pos = source_positions[nearest_idx]
        
        # Display the matched position
        if positions_in_degrees:
            print(f"Using HRTF at position: Az={nearest_pos[0]:.1f}°, El={nearest_pos[1]:.1f}°")
        else:
            print(f"Using HRTF at position: Az={np.rad2deg(nearest_pos[0]):.1f}°, El={np.rad2deg(nearest_pos[1]):.1f}°")
        
        return nearest_idx, nearest_pos
        
    except Exception as e:
        print(f"Error finding frontal HRTF: {e}")
        import traceback
        traceback.print_exc()
        return 0, None

def extract_hrtf_filters(hrtf, idx):
    """Extract left and right ear HRTF impulse responses"""
    hrtf_data = hrtf.Data_IR
    
    # Check for CTF (same filter for both ears)
    is_ctf = "CTF" in str(hrtf.GLOBAL_SOFAConventions)
    
    if is_ctf or (hrtf_data.shape[0] <= 2 and hrtf_data.shape[1] == 1):
        print("Using same filter for both ears (CTF)")
        try:
            common_filter = hrtf_data[0, 0, :]
            return common_filter, common_filter
        except:
            # Fallback: simple impulse
            dummy_ir = np.zeros(256)
            dummy_ir[0] = 1.0
            return dummy_ir, dummy_ir
    
    # Regular HRTF with separate left/right filters
    try:
        # Try common dimension ordering
        if hrtf_data.shape[0] == 2 and hrtf_data.shape[1] > 2:
            # Shape: (R=2, M=positions, N=samples)
            hrtf_left = hrtf_data[0, idx, :]
            hrtf_right = hrtf_data[1, idx, :]
        else:
            # Shape: (M=positions, R=2, N=samples)
            hrtf_left = hrtf_data[idx, 0, :]
            hrtf_right = hrtf_data[idx, 1, :]
        
        return hrtf_left, hrtf_right
        
    except Exception as e:
        print(f"Error extracting HRTF filters: {e}")
        # Fallback
        dummy_ir = np.zeros(256)
        dummy_ir[0] = 1.0
        return dummy_ir, dummy_ir

#############################################################
# LOOMING STIMULUS GENERATION
#############################################################

def create_amplitude_envelope(duration, fs, initial_dist, final_dist, use_inverse_square=True):
    """
    Create amplitude envelope for approaching sound based on distance
    
    For frontal looming, the primary cue is increasing amplitude.
    Uses inverse square law: amplitude ∝ 1/distance
    (intensity ∝ 1/distance^2, but we perceive amplitude/pressure)
    """
    num_samples = int(duration * fs)
    
    # Create smooth distance trajectory (linear movement)
    distances = np.linspace(initial_dist, final_dist, num_samples)
    
    if use_inverse_square:
        # Inverse distance law (physically accurate)
        # Amplitude is inversely proportional to distance
        amplitudes = 1.0 / distances
    else:
        # Linear amplitude increase (less realistic but smoother)
        amplitudes = np.linspace(1.0/initial_dist, 1.0/final_dist, num_samples)
    
    # Normalize so maximum amplitude is 1.0
    amplitudes = amplitudes / np.max(amplitudes)
    
    # Apply gentle fade in/out to avoid clicks
    fade_samples = int(0.05 * fs)  # 50ms fade
    
    # Fade in at start
    fade_in = np.linspace(0, 1, fade_samples)
    amplitudes[:fade_samples] *= fade_in
    
    # Fade out at end
    fade_out = np.linspace(1, 0, fade_samples)
    amplitudes[-fade_samples:] *= fade_out
    
    return amplitudes, distances

def generate_looming_stimulus(hrtf, noise_type, verbose=True):
    """
    Generate a frontal looming stimulus
    
    Processing chain:
    1. Generate noise signal
    2. Apply frontal HRTF (spatial filtering)
    3. Apply distance-based amplitude envelope
    4. Keep both ears identical (no ITD/ILD for frontal approach)
    """
    if verbose:
        print(f"\nGenerating looming stimulus with {noise_type} noise...")
    
    # Step 1: Generate source noise
    source_signal = generate_noise(noise_type, DURATION, SAMPLE_RATE)
    
    # Step 2: Find and apply frontal HRTF
    nearest_idx, nearest_pos = find_frontal_hrtf(hrtf, AZIMUTH_ANGLE, ELEVATION_ANGLE)
    hrtf_left, hrtf_right = extract_hrtf_filters(hrtf, nearest_idx)
    
    if verbose:
        print("Applying frontal HRTF filters...")
    
    # Apply HRTF filters (spectral coloring from head/pinnae)
    filtered_left = signal.fftconvolve(source_signal, hrtf_left, mode='same')
    filtered_right = signal.fftconvolve(source_signal, hrtf_right, mode='same')
    
    # Step 3: Create distance-based amplitude envelope
    if verbose:
        print("Applying distance-based amplitude envelope...")
    
    amplitude_envelope, distances = create_amplitude_envelope(
        DURATION, SAMPLE_RATE, INITIAL_DISTANCE, FINAL_DISTANCE, USE_INVERSE_SQUARE_LAW
    )
    
    # Step 4: Apply amplitude envelope to both channels
    # For frontal approach, both ears receive identical amplitude changes
    output_left = filtered_left * amplitude_envelope
    output_right = filtered_right * amplitude_envelope
    
    # Combine into stereo signal
    output_signal = np.column_stack((output_left, output_right))
    
    # Normalize and apply volume control
    max_val = np.max(np.abs(output_signal))
    if max_val > 0:
        output_signal = output_signal / max_val * 0.95  # Avoid clipping
        output_signal = output_signal * VOLUME_LEVEL
    
    if verbose:
        print(f"Generated {len(output_signal)/SAMPLE_RATE:.2f}s looming stimulus")
    
    return output_signal, distances

#############################################################
# VISUALIZATION
#############################################################

def plot_looming_trajectory(distances, noise_type, output_path):
    """Create visualization of the looming trajectory"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    time_axis = np.linspace(0, DURATION, len(distances))
    
    # Left plot: Distance over time
    ax1.plot(time_axis, distances, 'b-', linewidth=2.5, label='Distance trajectory')
    ax1.fill_between(time_axis, distances, alpha=0.3)
    
    # Mark start and end
    ax1.scatter(time_axis[0], distances[0], color='green', s=150, 
                marker='o', label='Start (far)', zorder=5, edgecolor='darkgreen', linewidth=2)
    ax1.scatter(time_axis[-1], distances[-1], color='red', s=150, 
                marker='o', label='End (near)', zorder=5, edgecolor='darkred', linewidth=2)
    
    ax1.grid(True, linestyle='--', alpha=0.4)
    ax1.set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Distance (meters)', fontsize=12, fontweight='bold')
    ax1.set_title(f'Frontal Looming: {noise_type.capitalize()} Noise', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10, loc='upper right')
    
    # Add velocity annotation
    velocity = (INITIAL_DISTANCE - FINAL_DISTANCE) / DURATION
    ax1.text(0.02, 0.98, f'Approach velocity: {velocity:.2f} m/s', 
             transform=ax1.transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Right plot: Top-down view
    ax2.set_aspect('equal')
    
    # Draw listener (head and ears)
    head_circle = plt.Circle((0, 0), 0.09, fill=False, color='blue', linewidth=2, label='Listener (head)')
    ax2.add_artist(head_circle)
    ax2.scatter([0], [0], color='blue', s=100, marker='o', zorder=5)
    ax2.scatter([-0.09, 0.09], [0, 0], color='blue', s=80, marker='s', label='Ears')
    
    # Draw source trajectory (approaching from front)
    # Front is along positive y-axis
    source_x = np.zeros_like(distances)
    source_y = distances  # Approaching along y-axis
    
    # Plot trajectory
    scatter = ax2.scatter(source_x, source_y, c=time_axis, cmap='plasma', 
                         s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
    ax2.plot(source_x, source_y, 'k--', alpha=0.3, linewidth=1)
    
    # Mark start and end positions
    ax2.scatter(source_x[0], source_y[0], color='green', s=200, marker='o', 
                edgecolor='darkgreen', linewidth=2, zorder=10, label='Start')
    ax2.scatter(source_x[-1], source_y[-1], color='red', s=200, marker='o', 
                edgecolor='darkred', linewidth=2, zorder=10, label='End')
    
    # Add arrow showing direction
    arrow_idx = len(distances) // 2
    ax2.annotate('', xy=(source_x[arrow_idx+10], source_y[arrow_idx+10]),
                xytext=(source_x[arrow_idx], source_y[arrow_idx]),
                arrowprops=dict(arrowstyle='->', lw=2, color='red'))
    
    # Colorbar for time
    cbar = plt.colorbar(scatter, ax=ax2)
    cbar.set_label('Time (seconds)', fontsize=10)
    
    ax2.grid(True, linestyle='--', alpha=0.3)
    ax2.set_xlabel('X position (meters)', fontsize=11)
    ax2.set_ylabel('Y position (meters)', fontsize=11)
    ax2.set_title('Top-Down View (Front = +Y)', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9, loc='upper left')
    
    # Set equal axis limits
    max_range = max(abs(INITIAL_DISTANCE), abs(FINAL_DISTANCE)) * 1.3
    ax2.set_xlim(-max_range, max_range)
    ax2.set_ylim(-0.2, max_range)
    
    # Add "FRONT" label
    ax2.text(0, max_range * 0.9, 'FRONT', fontsize=12, fontweight='bold', 
             ha='center', color='darkblue')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Saved trajectory plot: {os.path.basename(output_path)}")

#############################################################
# MAIN EXECUTION
#############################################################

def main():
    """Generate frontal looming stimuli for all noise types"""
    
    print("=" * 70)
    print("FRONTAL LOOMING STIMULUS GENERATOR")
    print("=" * 70)
    
    # Check dependencies
    if not SOFA_AVAILABLE:
        print("\nERROR: sofar library is required.")
        print("Install with: pip install sofar --break-system-packages")
        return
    
    # Check SOFA file exists
    if not os.path.exists(SOFA_FILE):
        print(f"\nERROR: SOFA file not found: {SOFA_FILE}")
        return
    
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"\nOutput directory: {OUTPUT_DIR}")
    
    # Load HRTF
    print("\n" + "-" * 70)
    hrtf = load_sofa_file(SOFA_FILE)
    
    if hrtf is None:
        print("ERROR: Failed to load SOFA file")
        return
    
    # Generate stimuli for each noise type
    print("\n" + "-" * 70)
    print("GENERATING LOOMING STIMULI")
    print("-" * 70)
    
    generated_files = []
    
    for idx, noise_type in enumerate(NOISE_TYPES, start=1):
        print(f"\n[{idx}/{len(NOISE_TYPES)}] Processing {noise_type} noise...")
        
        try:
            # Generate looming stimulus
            output_signal, distances = generate_looming_stimulus(hrtf, noise_type, verbose=True)
            
            # Save audio file
            filename = f"Loom-{idx}-{noise_type}.wav"
            output_path = os.path.join(OUTPUT_DIR, filename)
            sf.write(output_path, output_signal, SAMPLE_RATE)
            print(f"✓ Saved: {filename}")
            generated_files.append(filename)
            
            # Create visualization
            plot_filename = f"Loom-{idx}-{noise_type}_trajectory.png"
            plot_path = os.path.join(OUTPUT_DIR, plot_filename)
            plot_looming_trajectory(distances, noise_type, plot_path)
            
        except Exception as e:
            print(f"✗ ERROR generating {noise_type} stimulus: {e}")
            import traceback
            traceback.print_exc()
    
    # Summary
    print("\n" + "=" * 70)
    print("GENERATION COMPLETE")
    print("=" * 70)
    print(f"\nGenerated {len(generated_files)} looming stimuli:")
    for filename in generated_files:
        print(f"  • {filename}")
    
    print(f"\nParameters used:")
    print(f"  • Initial distance: {INITIAL_DISTANCE} m")
    print(f"  • Final distance: {FINAL_DISTANCE} m")
    print(f"  • Duration: {DURATION} s")
    print(f"  • Approach velocity: {(INITIAL_DISTANCE - FINAL_DISTANCE) / DURATION:.2f} m/s")
    print(f"  • Azimuth: {AZIMUTH_ANGLE}° (frontal)")
    print(f"  • Elevation: {ELEVATION_ANGLE}° (ear level)")
    print(f"  • Sample rate: {SAMPLE_RATE} Hz")
    print(f"  • Volume level: {VOLUME_LEVEL:.1f}")
    print(f"  • Distance model: {'Inverse square law' if USE_INVERSE_SQUARE_LAW else 'Linear'}")
    print(f"  • HRTF model: FABIAN (neutral head orientation)")
    
    print(f"\nAll files saved to:")
    print(f"  {OUTPUT_DIR}")
    print("\n" + "=" * 70)

if __name__ == "__main__":
    main()

FRONTAL LOOMING STIMULUS GENERATOR

Output directory: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\LoomingStimuli

----------------------------------------------------------------------
Loading SOFA file: FABIAN_HRIR_measured_HATO_0.sofa
SOFA Convention: SimpleFreeFieldHRIR v1.0
HRTF shape: (11950, 2, 256)

----------------------------------------------------------------------
GENERATING LOOMING STIMULI
----------------------------------------------------------------------

[1/4] Processing pink noise...

Generating looming stimulus with pink noise...
Using HRTF at position: Az=0.0°, El=0.0°
Applying frontal HRTF filters...
Applying distance-based amplitude envelope...
Generated 3.00s looming stimulus
✓ Saved: Loom-1-pink.wav
Saved trajectory plot: Loom-1-pink_trajectory.png

[2/4] Processing blue noise...

Generating looming stimulus with blue noise...
Using HRTF at position: Az=0.0°, El=0.0°
Applying frontal HRTF filters...
Applying distance-based amplitude envelo

In [5]:
"""
========================================================================
LOOMING X TACTILE STIMULI COMBINATION SCRIPT (JUPYTER VERSION)
========================================================================
This script creates stereo audio files by combining:
  LEFT CHANNEL:  Tactile stimuli
  RIGHT CHANNEL: Looming stimuli (newly generated)

Creates 20 combined files (5 tactile × 4 looming)
========================================================================
"""

import numpy as np
import soundfile as sf
import os
from pathlib import Path

print("=" * 70)
print("LOOMING X TACTILE STIMULI GENERATOR")
print("=" * 70)

# ========================================================================
# INPUT FILES
# ========================================================================

# Tactile stimuli (LEFT channel)
TACTILE_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\TactileStimuli"
tactile_files = [
    "tactile_stimulus_300ms.wav",
    "tactile_stimulus_800ms.wav",
    "tactile_stimulus_1500ms.wav",
    "tactile_stimulus_2200ms.wav",
    "tactile_stimulus_2700ms.wav"
]

# Looming stimuli (RIGHT channel) - newly generated
LOOMING_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\LoomingStimuli"
looming_files = [
    "Loom-1-pink.wav",
    "Loom-2-blue.wav",
    "Loom-3-white.wav",
    "Loom-4-brown.wav"
]

# Output directory
OUTPUT_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\LoomingXTactile_Stimuli"

# ========================================================================
# SETUP
# ========================================================================

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nInput Files:")
print(f"  Tactile (LEFT):  {len(tactile_files)} files")
print(f"  Looming (RIGHT): {len(looming_files)} files")
print(f"\nOutput Directory:")
print(f"  {OUTPUT_DIR}")
print(f"\nGenerating {len(tactile_files) * len(looming_files)} combined stereo files...")
print("-" * 70)

# ========================================================================
# HELPER FUNCTION
# ========================================================================

def combine_stereo(left_file, right_file, output_file):
    """
    Combine two mono audio files into stereo
    LEFT channel: left_file
    RIGHT channel: right_file
    """
    try:
        # Read left channel (tactile)
        left_audio, left_sr = sf.read(left_file)
        
        # Read right channel (looming)
        right_audio, right_sr = sf.read(right_file)
        
        # Verify sample rates match
        if left_sr != right_sr:
            raise ValueError(f"Sample rate mismatch: {left_sr} vs {right_sr}")
        
        # If audio is already stereo, take first channel only
        if len(left_audio.shape) > 1:
            left_audio = left_audio[:, 0]
        if len(right_audio.shape) > 1:
            right_audio = right_audio[:, 0]
        
        # Make both arrays the same length (pad shorter one with zeros)
        max_len = max(len(left_audio), len(right_audio))
        
        if len(left_audio) < max_len:
            left_audio = np.pad(left_audio, (0, max_len - len(left_audio)))
        if len(right_audio) < max_len:
            right_audio = np.pad(right_audio, (0, max_len - len(right_audio)))
        
        # Create stereo array
        stereo_audio = np.column_stack((left_audio, right_audio))
        
        # Write output file
        sf.write(output_file, stereo_audio, left_sr)
        
        return True, None
        
    except Exception as e:
        return False, str(e)

# ========================================================================
# GENERATE COMBINATIONS
# ========================================================================

count = 0
total = len(tactile_files) * len(looming_files)
successful = 0
failed = 0
failed_files = []

for tactile_file in tactile_files:
    for looming_file in looming_files:
        count += 1
        
        # Get file names without extensions
        tactile_name = Path(tactile_file).stem
        looming_name = Path(looming_file).stem
        
        # Create full paths
        tactile_path = os.path.join(TACTILE_DIR, tactile_file)
        looming_path = os.path.join(LOOMING_DIR, looming_file)
        
        # Create output filename
        output_filename = f"Stereo_LEFT_{tactile_name}__RIGHT_{looming_name}.wav"
        output_path = os.path.join(OUTPUT_DIR, output_filename)
        
        print(f"\n[{count}/{total}] {output_filename}")
        
        # Check if input files exist
        if not os.path.exists(tactile_path):
            print(f"   ✗ ERROR: Tactile file not found: {tactile_file}")
            failed += 1
            failed_files.append(output_filename)
            continue
            
        if not os.path.exists(looming_path):
            print(f"   ✗ ERROR: Looming file not found: {looming_file}")
            failed += 1
            failed_files.append(output_filename)
            continue
        
        # Combine files
        success, error_msg = combine_stereo(tactile_path, looming_path, output_path)
        
        if success:
            print(f"   ✓ Success")
            successful += 1
        else:
            print(f"   ✗ ERROR: {error_msg}")
            failed += 1
            failed_files.append(output_filename)

# ========================================================================
# SUMMARY
# ========================================================================

print("\n" + "=" * 70)
print("COMPLETE!")
print("=" * 70)

print(f"\nResults:")
print(f"  ✓ Successful: {successful}/{total}")
print(f"  ✗ Failed:     {failed}/{total}")

if failed > 0:
    print(f"\nFailed files:")
    for filename in failed_files:
        print(f"  • {filename}")

print(f"\nOutput location:")
print(f"  {OUTPUT_DIR}")

print(f"\nFile naming convention:")
print(f"  Stereo_LEFT_[tactile]__RIGHT_[looming].wav")

print(f"\nChannel layout:")
print(f"  LEFT:  Tactile stimulus")
print(f"  RIGHT: Looming stimulus (frontal approach with HRTF)")

print("\n" + "=" * 70)

LOOMING X TACTILE STIMULI GENERATOR

Input Files:
  Tactile (LEFT):  5 files
  Looming (RIGHT): 4 files

Output Directory:
  G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\LoomingXTactile_Stimuli

Generating 20 combined stereo files...
----------------------------------------------------------------------

[1/20] Stereo_LEFT_tactile_stimulus_300ms__RIGHT_Loom-1-pink.wav
   ✓ Success

[2/20] Stereo_LEFT_tactile_stimulus_300ms__RIGHT_Loom-2-blue.wav
   ✓ Success

[3/20] Stereo_LEFT_tactile_stimulus_300ms__RIGHT_Loom-3-white.wav
   ✓ Success

[4/20] Stereo_LEFT_tactile_stimulus_300ms__RIGHT_Loom-4-brown.wav
   ✓ Success

[5/20] Stereo_LEFT_tactile_stimulus_800ms__RIGHT_Loom-1-pink.wav
   ✓ Success

[6/20] Stereo_LEFT_tactile_stimulus_800ms__RIGHT_Loom-2-blue.wav
   ✓ Success

[7/20] Stereo_LEFT_tactile_stimulus_800ms__RIGHT_Loom-3-white.wav
   ✓ Success

[8/20] Stereo_LEFT_tactile_stimulus_800ms__RIGHT_Loom-4-brown.wav
   ✓ Success

[9/20] Stereo_LEFT_tactile_stimulus_15

Generates AUDIO-TACTILE TRIALS (120 trials):
5 SOAs × 4 Noise Types × 2 Respiratory Phases × 3 reps = 120
├─ 300ms × (Pink, Blue, White, Brown) × (Inhale, Exhale) × 3 = 24
├─ 800ms × (Pink, Blue, White, Brown) × (Inhale, Exhale) × 3 = 24
├─ 1500ms × (Pink, Blue, White, Brown) × (Inhale, Exhale) × 3 = 24
├─ 2200ms × (Pink, Blue, White, Brown) × (Inhale, Exhale) × 3 = 24
└─ 2700ms × (Pink, Blue, White, Brown) × (Inhale, Exhale) × 3 = 24


5 SOA × 4 noise × 2 phase = 40 cells × 3 reps = 120 audio-tactile trials
Inhale/
  ├─ ...300ms...pink_rep1.wav
  ├─ ...300ms...pink_rep2.wav
  ├─ ...300ms...pink_rep3.wav
  ├─ ...300ms...blue_rep1.wav
  └─ ... (60 files total)

Exhale/
  ├─ ...300ms...pink_rep1.wav
  ├─ ...300ms...pink_rep2.wav
  ├─ ...300ms...pink_rep3.wav
  └─ ... (60 files total)

In [40]:
"""
========================================================================
BREATHING INSTRUCTIONS + LOOMING-TACTILE STIMULI COMBINER
========================================================================

This script combines box breathing instructions with looming-tactile stimuli:

1. Loads breathing instructions from MP3 files
2. Extends looming-tactile files from 3000ms to 4000ms (add 500ms before + 500ms after)
3. Converts breathing instructions to RIGHT stereo channel
4. Prepends breathing instructions to extended stimuli
5. Creates separate Inhale and Exhale versions
6. Saves as lossless WAV format for maximum quality

Stereo channel routing:
   - LEFT channel: Silence → Tactile trigger
   - RIGHT channel: Breathing instructions → Looming audio

Timeline per trial (8 seconds total):
- 0-4s: Breathing instruction (4s inhale/exhale)
- 4-8s: Breath hold (4s) with looming + tactile stimuli (padded to 4s)

Input: MP3 (breathing), WAV (looming-tactile)
Output: WAV (lossless, combined)

========================================================================
"""

import numpy as np
import soundfile as sf
from pydub import AudioSegment
from scipy import signal as sp_signal
import os
from pathlib import Path

print("=" * 70)
print("BREATHING INSTRUCTIONS + LOOMING-TACTILE COMBINER")
print("=" * 70)

# ========================================================================
# FILE PATHS
# ========================================================================

# Breathing instruction files (MP3 input)
INHALE_INSTRUCTION = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\0. BoxBreathingInstructions\Inhale-2-3-4-hold_v3_FIXED.mp3"
EXHALE_INSTRUCTION = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\0. BoxBreathingInstructions\Exhale-2-3-4-hold_FIXED.mp3"

# Input directory with looming-tactile stereo files
INPUT_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\4. LoomingXTactile_Stimuli"

# Output directories
OUTPUT_DIR_INHALE = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\5. Breathingphase+LoomingXTactile\Inhale"
OUTPUT_DIR_EXHALE = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\5. Breathingphase+LoomingXTactile\Exhale"

# Looming-tactile files to process
LOOMING_TACTILE_FILES = [
    "Stereo_LEFT_tactile_stimulus_300ms__RIGHT_Loom-1-pink.wav",
    "Stereo_LEFT_tactile_stimulus_300ms__RIGHT_Loom-2-blue.wav",
    "Stereo_LEFT_tactile_stimulus_300ms__RIGHT_Loom-3-white.wav",
    "Stereo_LEFT_tactile_stimulus_300ms__RIGHT_Loom-4-brown.wav",
    "Stereo_LEFT_tactile_stimulus_800ms__RIGHT_Loom-1-pink.wav",
    "Stereo_LEFT_tactile_stimulus_800ms__RIGHT_Loom-2-blue.wav",
    "Stereo_LEFT_tactile_stimulus_800ms__RIGHT_Loom-3-white.wav",
    "Stereo_LEFT_tactile_stimulus_800ms__RIGHT_Loom-4-brown.wav",
    "Stereo_LEFT_tactile_stimulus_1500ms__RIGHT_Loom-1-pink.wav",
    "Stereo_LEFT_tactile_stimulus_1500ms__RIGHT_Loom-2-blue.wav",
    "Stereo_LEFT_tactile_stimulus_1500ms__RIGHT_Loom-3-white.wav",
    "Stereo_LEFT_tactile_stimulus_1500ms__RIGHT_Loom-4-brown.wav",
    "Stereo_LEFT_tactile_stimulus_2200ms__RIGHT_Loom-1-pink.wav",
    "Stereo_LEFT_tactile_stimulus_2200ms__RIGHT_Loom-2-blue.wav",
    "Stereo_LEFT_tactile_stimulus_2200ms__RIGHT_Loom-3-white.wav",
    "Stereo_LEFT_tactile_stimulus_2200ms__RIGHT_Loom-4-brown.wav",
    "Stereo_LEFT_tactile_stimulus_2700ms__RIGHT_Loom-1-pink.wav",
    "Stereo_LEFT_tactile_stimulus_2700ms__RIGHT_Loom-2-blue.wav",
    "Stereo_LEFT_tactile_stimulus_2700ms__RIGHT_Loom-3-white.wav",
    "Stereo_LEFT_tactile_stimulus_2700ms__RIGHT_Loom-4-brown.wav",
]

# Timing parameters
TARGET_SAMPLE_RATE = 44100  # Standard sample rate

# ========================================================================
# HELPER FUNCTIONS
# ========================================================================

def load_mp3_as_stereo_right(mp3_path, target_sr=44100):
    """
    Load MP3 file and convert to stereo with audio in RIGHT channel only
    
    Returns: numpy array (N, 2) and sample rate
    """
    print(f"  Loading MP3: {os.path.basename(mp3_path)}")
    
    # Check if file exists
    if not os.path.exists(mp3_path):
        raise FileNotFoundError(f"MP3 file not found: {mp3_path}")
    
    try:
        # Load MP3 using pydub
        audio = AudioSegment.from_mp3(mp3_path)
        
        # Convert to target sample rate
        if audio.frame_rate != target_sr:
            print(f"    Resampling from {audio.frame_rate} Hz to {target_sr} Hz...")
            audio = audio.set_frame_rate(target_sr)
        
        # Convert to mono if stereo (take first channel)
        if audio.channels > 1:
            audio = audio.set_channels(1)
        
        # Get samples as numpy array
        samples = np.array(audio.get_array_of_samples(), dtype=np.float32)
        
        # Normalize to [-1, 1]
        samples = samples / (2**15)
        
        # Create stereo array with audio in RIGHT channel only
        stereo_right = np.zeros((len(samples), 2), dtype=np.float32)
        stereo_right[:, 0] = 0.0        # LEFT channel = silence
        stereo_right[:, 1] = samples    # RIGHT channel = breathing instruction
        
        print(f"    Duration: {len(stereo_right) / target_sr:.2f}s")
        print(f"    Sample rate: {target_sr} Hz")
        print(f"    Channels: LEFT=silence, RIGHT=instruction")
        
        return stereo_right, target_sr
        
    except Exception as e:
        print(f"    ✗ Error reading MP3 file: {e}")
        print(f"    Full path: {mp3_path}")
        raise

def extend_looming_tactile_to_exact_duration(audio, sr, breathing_duration, target_total_duration=8.0):
    """
    Extend looming-tactile stimulus to create exactly target_total_duration
    
    Calculates padding needed to make total file exactly 8.0 seconds after combining
    with breathing instruction.
    
    Input audio is stereo:
    - LEFT channel: Tactile trigger
    - RIGHT channel: Looming audio
    
    Returns extended stereo audio with calculated padding
    """
    # Calculate how long the stimulus portion should be
    stimulus_duration_needed = target_total_duration - breathing_duration
    
    # Calculate how many samples that is
    stimulus_samples_needed = int(stimulus_duration_needed * sr)
    
    # Current audio length
    current_samples = len(audio)
    
    # Total padding needed
    total_padding_needed = stimulus_samples_needed - current_samples
    
    # Split padding evenly before and after (with rounding)
    pad_start_samples = total_padding_needed // 2
    pad_end_samples = total_padding_needed - pad_start_samples  # Ensures exact total
    
    # Create silence padding (stereo)
    silence_start = np.zeros((pad_start_samples, 2))
    silence_end = np.zeros((pad_end_samples, 2))
    
    # Concatenate: silence + audio + silence
    extended = np.vstack([silence_start, audio, silence_end])
    
    # Verify exact length
    actual_duration = len(extended) / sr
    
    return extended, pad_start_samples, pad_end_samples, actual_duration

def combine_breathing_and_stimulus(breathing_audio, stimulus_audio, sr):
    """
    Combine breathing instruction with extended looming-tactile stimulus
    
    Breathing audio: (N, 2) - LEFT=silence, RIGHT=instruction
    Stimulus audio: (M, 2) - LEFT=tactile, RIGHT=looming
    
    Result: (N+M, 2) - LEFT=tactile, RIGHT=instruction+looming
    """
    # Concatenate vertically (time dimension)
    combined = np.vstack([breathing_audio, stimulus_audio])
    
    # Verify duration
    actual_duration = len(combined) / sr
    
    return combined, actual_duration

# ========================================================================
# CREATE OUTPUT DIRECTORIES
# ========================================================================

os.makedirs(OUTPUT_DIR_INHALE, exist_ok=True)
os.makedirs(OUTPUT_DIR_EXHALE, exist_ok=True)

print(f"\nOutput directories:")
print(f"  Inhale: {OUTPUT_DIR_INHALE}")
print(f"  Exhale: {OUTPUT_DIR_EXHALE}")

# ========================================================================
# LOAD BREATHING INSTRUCTIONS
# ========================================================================

print("\n" + "-" * 70)
print("LOADING BREATHING INSTRUCTIONS")
print("-" * 70)

print("\n[1/2] Loading INHALE instruction...")
inhale_instruction, inhale_sr = load_mp3_as_stereo_right(INHALE_INSTRUCTION, TARGET_SAMPLE_RATE)

print("\n[2/2] Loading EXHALE instruction...")
exhale_instruction, exhale_sr = load_mp3_as_stereo_right(EXHALE_INSTRUCTION, TARGET_SAMPLE_RATE)

# ========================================================================
# PROCESS EACH LOOMING-TACTILE FILE
# ========================================================================

print("\n" + "=" * 70)
print("PROCESSING LOOMING-TACTILE STIMULI")
print("=" * 70)

total_files = len(LOOMING_TACTILE_FILES)
success_count = 0
error_count = 0
errors = []

for idx, filename in enumerate(LOOMING_TACTILE_FILES, start=1):
    print(f"\n[{idx}/{total_files}] Processing: {filename}")
    
    try:
        # Full input path
        input_path = os.path.join(INPUT_DIR, filename)
        
        # Check if file exists
        if not os.path.exists(input_path):
            print(f"  ✗ ERROR: File not found")
            error_count += 1
            errors.append(f"{filename} - File not found")
            continue
        
        # Load looming-tactile stereo file
        print(f"  Loading stimulus...")
        stimulus_audio, stimulus_sr = sf.read(input_path)
        
        # Verify it's stereo
        if len(stimulus_audio.shape) != 2 or stimulus_audio.shape[1] != 2:
            print(f"  ✗ ERROR: Not a stereo file")
            error_count += 1
            errors.append(f"{filename} - Not stereo")
            continue
        
        # Resample if needed
        if stimulus_sr != TARGET_SAMPLE_RATE:
            print(f"  Resampling from {stimulus_sr} Hz to {TARGET_SAMPLE_RATE} Hz...")
            # Simple resampling (for more accurate, use scipy.signal.resample)
            ratio = TARGET_SAMPLE_RATE / stimulus_sr
            new_length = int(len(stimulus_audio) * ratio)
            stimulus_audio = np.array([
                np.interp(
                    np.linspace(0, len(stimulus_audio), new_length),
                    np.arange(len(stimulus_audio)),
                    stimulus_audio[:, ch]
                ) for ch in range(2)
            ]).T
            stimulus_sr = TARGET_SAMPLE_RATE
        
        original_duration = len(stimulus_audio) / stimulus_sr
        print(f"    Original duration: {original_duration * 1000:.0f} ms")
        
        # Get breathing instruction duration
        breathing_duration = len(inhale_instruction) / TARGET_SAMPLE_RATE
        print(f"    Breathing duration: {breathing_duration:.3f} s")
        
        # Extend stimulus to make total exactly 8.0 seconds
        print(f"  Extending stimulus to create exactly 8.0s total...")
        extended_stimulus, pad_start, pad_end, actual_stim_dur = extend_looming_tactile_to_exact_duration(
            stimulus_audio, 
            stimulus_sr,
            breathing_duration,
            target_total_duration=8.0
        )
        
        print(f"    Padding: {pad_start} samples before, {pad_end} samples after")
        print(f"    Stimulus portion duration: {actual_stim_dur:.3f} s")
        
        # Create INHALE version (single file)
        print(f"  Creating INHALE version...")
        inhale_combined, inhale_total_duration = combine_breathing_and_stimulus(
            inhale_instruction, 
            extended_stimulus, 
            TARGET_SAMPLE_RATE
        )
        
        # Output filename
        base_name = Path(filename).stem  # Get filename without extension
        inhale_filename = f"{base_name}.wav"
        inhale_output_path = os.path.join(OUTPUT_DIR_INHALE, inhale_filename)
        sf.write(inhale_output_path, inhale_combined, TARGET_SAMPLE_RATE)
        
        print(f"    ✓ Saved: {inhale_filename} ({inhale_total_duration:.3f}s)")
        
        # Create EXHALE version (single file)
        print(f"  Creating EXHALE version...")
        exhale_combined, exhale_total_duration = combine_breathing_and_stimulus(
            exhale_instruction, 
            extended_stimulus, 
            TARGET_SAMPLE_RATE
        )
        
        # Output filename
        exhale_filename = f"{base_name}.wav"
        exhale_output_path = os.path.join(OUTPUT_DIR_EXHALE, exhale_filename)
        sf.write(exhale_output_path, exhale_combined, TARGET_SAMPLE_RATE)
        
        print(f"    ✓ Saved: {exhale_filename} ({exhale_total_duration:.3f}s)")
        
        success_count += 1
        
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        error_count += 1
        errors.append(f"{filename} - {str(e)}")
        import traceback
        traceback.print_exc()

# ========================================================================
# SUMMARY
# ========================================================================

print("\n" + "=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)

print(f"\nResults:")
print(f"  ✓ Successfully processed: {success_count}/{total_files} files")
print(f"  ✗ Errors: {error_count}/{total_files} files")

if error_count > 0:
    print(f"\nErrors encountered:")
    for error in errors:
        print(f"  • {error}")

print(f"\nOutput files created:")
print(f"  Inhale versions: {OUTPUT_DIR_INHALE}")
print(f"  Exhale versions: {OUTPUT_DIR_EXHALE}")
print(f"  Total files: {success_count * 2} ({success_count} per phase)")
print(f"    - {success_count} unique stimuli × 2 phases")

print(f"\nTrial structure (per file):")
print(f"  0.0 - ~4.0s: Breathing instruction (inhale/exhale)")
print(f"  ~4.0 - 8.0s: Breath hold with looming + tactile")
print(f"    Padding: Dynamic (adjusted for exact 8.0s total)")
print(f"    Contains: 3000ms looming audio + tactile stimulus")
print(f"  Total: EXACTLY 8.000 seconds per trial")

print(f"\nStereo channel layout:")
print(f"  LEFT:  Silence → Tactile trigger signal")
print(f"  RIGHT: Breathing instruction → Looming audio")

print("\n" + "=" * 70)

BREATHING INSTRUCTIONS + LOOMING-TACTILE COMBINER

Output directories:
  Inhale: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\5. Breathingphase+LoomingXTactile\Inhale
  Exhale: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\5. Breathingphase+LoomingXTactile\Exhale

----------------------------------------------------------------------
LOADING BREATHING INSTRUCTIONS
----------------------------------------------------------------------

[1/2] Loading INHALE instruction...
  Loading MP3: Inhale-2-3-4-hold_v3_FIXED.mp3
    Duration: 4.04s
    Sample rate: 44100 Hz
    Channels: LEFT=silence, RIGHT=instruction

[2/2] Loading EXHALE instruction...
  Loading MP3: Exhale-2-3-4-hold_FIXED.mp3
    Duration: 4.04s
    Sample rate: 44100 Hz
    Channels: LEFT=silence, RIGHT=instruction

PROCESSING LOOMING-TACTILE STIMULI

[1/20] Processing: Stereo_LEFT_tactile_stimulus_300ms__RIGHT_Loom-1-pink.wav
  Loading stimulus...
    Original duration: 3000 ms
    Breat

AUDIO-TACTILE TRIALS (120 trials):
5 SOAs × 4 Noise Types × 2 Respiratory Phases × 3 reps = 120
├─ 300ms × (Pink, Blue, White, Brown) × (Inhale, Exhale) × 3 = 24
├─ 800ms × (Pink, Blue, White, Brown) × (Inhale, Exhale) × 3 = 24
├─ 1500ms × (Pink, Blue, White, Brown) × (Inhale, Exhale) × 3 = 24
├─ 2200ms × (Pink, Blue, White, Brown) × (Inhale, Exhale) × 3 = 24
└─ 2700ms × (Pink, Blue, White, Brown) × (Inhale, Exhale) × 3 = 24

BASELINE TRIALS (120 trials):
5 SOAs × 2 Respiratory Phases × 12 reps = 120
├─ 300ms × (Inhale, Exhale) × 12 = 24
├─ 800ms × (Inhale, Exhale) × 12 = 24
├─ 1500ms × (Inhale, Exhale) × 12 = 24
├─ 2200ms × (Inhale, Exhale) × 12 = 24
└─ 2700ms × (Inhale, Exhale) × 12 = 24

CATCH TRIALS (20 trials):
5 SOAs × 2 Respiratory Phases × 2 reps = 20
(Noise type randomly assigned)

TOTAL: 260 trials

Baseline Tactile Trial Generator

In [22]:
"""
========================================================================
BASELINE TACTILE STIMULI GENERATOR (4 SECOND VERSION)
========================================================================

This script extends tactile-only stimuli to exactly 4000ms for baseline trials.

Process:
1. Load each tactile stimulus (with delay already included)
2. Add 500ms silence at the beginning
3. Calculate remaining time needed to reach 4000ms total
4. Add calculated silence at the end
5. Save as baseline tactile trial (exactly 4000ms)

These files will be used for baseline (tactile-only) trials in both
inhale and exhale conditions.

========================================================================
"""

import numpy as np
import soundfile as sf
import os
from pathlib import Path

print("=" * 70)
print("BASELINE TACTILE STIMULI GENERATOR (4 SECOND)")
print("=" * 70)

# ========================================================================
# FILE PATHS
# ========================================================================

# Input directory with tactile stimuli
INPUT_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\1. TactileStimuli"

# Output directory for baseline tactile trials
OUTPUT_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\1. TactileStimuli\BaselineTactileTrials"

# Tactile stimulus files to process
TACTILE_FILES = [
    "tactile_stimulus_300ms.wav",
    "tactile_stimulus_800ms.wav",
    "tactile_stimulus_1500ms.wav",
    "tactile_stimulus_2200ms.wav",
    "tactile_stimulus_2700ms.wav",
]

# Timing parameters
TARGET_DURATION_MS = 4000   # Target: exactly 4000ms
PREPEND_SILENCE_MS = 500    # Add 500ms at start
SAMPLE_RATE = 44100         # Standard sample rate

# ========================================================================
# CREATE OUTPUT DIRECTORY
# ========================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nOutput directory: {OUTPUT_DIR}")

# ========================================================================
# PROCESS EACH TACTILE STIMULUS
# ========================================================================

print("\n" + "=" * 70)
print("PROCESSING TACTILE STIMULI")
print("=" * 70)

success_count = 0
error_count = 0
errors = []

for idx, filename in enumerate(TACTILE_FILES, start=1):
    print(f"\n[{idx}/{len(TACTILE_FILES)}] Processing: {filename}")
    
    try:
        # Full input path
        input_path = os.path.join(INPUT_DIR, filename)
        
        # Check if file exists
        if not os.path.exists(input_path):
            print(f"  ✗ ERROR: File not found")
            error_count += 1
            errors.append(f"{filename} - File not found")
            continue
        
        # Load tactile stimulus
        print(f"  Loading stimulus...")
        audio, sr = sf.read(input_path)
        
        # Resample if needed
        if sr != SAMPLE_RATE:
            print(f"    Resampling from {sr} Hz to {SAMPLE_RATE} Hz...")
            from scipy import signal as sp_signal
            num_samples = int(len(audio) * SAMPLE_RATE / sr)
            if len(audio.shape) == 1:
                # Mono
                audio = sp_signal.resample(audio, num_samples)
            else:
                # Stereo
                audio = np.array([
                    sp_signal.resample(audio[:, ch], num_samples)
                    for ch in range(audio.shape[1])
                ]).T
            sr = SAMPLE_RATE
        
        # Ensure stereo
        if len(audio.shape) == 1:
            # Convert mono to stereo
            audio = np.column_stack([audio, audio])
            print(f"    Converted mono to stereo")
        
        original_duration_ms = len(audio) / sr * 1000
        print(f"    Original duration: {original_duration_ms:.1f} ms")
        
        # Calculate padding needed
        prepend_samples = int((PREPEND_SILENCE_MS / 1000) * sr)
        target_total_samples = int((TARGET_DURATION_MS / 1000) * sr)
        current_samples = len(audio)
        
        # Total padding needed
        total_padding_needed = target_total_samples - current_samples
        
        # We prepend 500ms, the rest goes at the end
        append_samples = total_padding_needed - prepend_samples
        
        print(f"  Padding calculation:")
        print(f"    Target total: {TARGET_DURATION_MS} ms ({target_total_samples} samples)")
        print(f"    Current: {original_duration_ms:.1f} ms ({current_samples} samples)")
        print(f"    Prepend: {PREPEND_SILENCE_MS} ms ({prepend_samples} samples)")
        print(f"    Append: {append_samples / sr * 1000:.1f} ms ({append_samples} samples)")
        
        # Create silence padding (stereo)
        silence_prepend = np.zeros((prepend_samples, 2))
        silence_append = np.zeros((append_samples, 2))
        
        # Concatenate: silence + audio + silence
        extended_audio = np.vstack([silence_prepend, audio, silence_append])
        
        # Verify exact duration
        final_duration_ms = len(extended_audio) / sr * 1000
        print(f"    Final duration: {final_duration_ms:.1f} ms")
        
        # Save extended stimulus
        output_filename = f"baseline_{filename}"
        output_path = os.path.join(OUTPUT_DIR, output_filename)
        sf.write(output_path, extended_audio, sr)
        
        print(f"  ✓ Saved: {output_filename}")
        
        # Verify it's exactly 4000ms
        if abs(final_duration_ms - TARGET_DURATION_MS) < 1.0:  # Within 1ms tolerance
            print(f"  ✓ Duration verified: EXACTLY {TARGET_DURATION_MS} ms")
        else:
            print(f"  ⚠ Duration warning: {final_duration_ms:.1f} ms (target: {TARGET_DURATION_MS} ms)")
        
        success_count += 1
        
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        error_count += 1
        errors.append(f"{filename} - {str(e)}")
        import traceback
        traceback.print_exc()

# ========================================================================
# SUMMARY
# ========================================================================

print("\n" + "=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)

print(f"\nResults:")
print(f"  ✓ Successfully processed: {success_count}/{len(TACTILE_FILES)} files")
print(f"  ✗ Errors: {error_count}/{len(TACTILE_FILES)} files")

if error_count > 0:
    print(f"\nErrors encountered:")
    for error in errors:
        print(f"  • {error}")

print(f"\nOutput files created:")
for filename in TACTILE_FILES:
    if os.path.exists(os.path.join(OUTPUT_DIR, f"baseline_{filename}")):
        print(f"  ✓ baseline_{filename}")

print(f"\nOutput location:")
print(f"  {OUTPUT_DIR}")

print(f"\nFile structure:")
print(f"  Duration: EXACTLY 4000 ms (4.0 seconds)")
print(f"  Format: Stereo WAV, 44100 Hz")
print(f"  Padding: 500ms prepended, remainder appended")

print(f"\nUsage:")
print(f"  These baseline tactile files are used for:")
print(f"  - Baseline-Inhale trials (tactile only, no audio)")
print(f"  - Baseline-Exhale trials (tactile only, no audio)")
print(f"  - Each stimulus will be combined with breathing instructions")
print(f"    to create 8-second baseline trials (4s breathe + 4s stimulus)")

print("\n" + "=" * 70)

BASELINE TACTILE STIMULI GENERATOR (4 SECOND)

Output directory: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\1. TactileStimuli\BaselineTactileTrials

PROCESSING TACTILE STIMULI

[1/5] Processing: tactile_stimulus_300ms.wav
  Loading stimulus...
    Converted mono to stereo
    Original duration: 400.0 ms
  Padding calculation:
    Target total: 4000 ms (176400 samples)
    Current: 400.0 ms (17640 samples)
    Prepend: 500 ms (22050 samples)
    Append: 3100.0 ms (136710 samples)
    Final duration: 4000.0 ms
  ✓ Saved: baseline_tactile_stimulus_300ms.wav
  ✓ Duration verified: EXACTLY 4000 ms

[2/5] Processing: tactile_stimulus_800ms.wav
  Loading stimulus...
    Converted mono to stereo
    Original duration: 900.0 ms
  Padding calculation:
    Target total: 4000 ms (176400 samples)
    Current: 900.0 ms (39690 samples)
    Prepend: 500 ms (22050 samples)
    Append: 2600.0 ms (114660 samples)
    Final duration: 4000.0 ms
  ✓ Saved: baseline_tactile_stimulus_800

Generate 8 sec baseline trials

========================================================================
5. BASELINE TRIALS GENERATOR (BREATHING + TACTILE ONLY)
========================================================================

In [41]:
"""
========================================================================
BASELINE TRIALS GENERATOR (BREATHING + TACTILE ONLY)
========================================================================

This script creates baseline trials by combining breathing instructions
with tactile-only stimuli (no looming audio).

Process:
1. Load breathing instructions (Inhale/Exhale) - ~4s each
2. Load 4-second baseline tactile stimuli (5 SOAs)
3. Combine to create 8-second baseline trials
4. Save single version per SOA per phase
5. Save to Inhale and Exhale baseline folders

Design:
- 5 SOAs × 2 phases = 10 baseline trial types
- 5 trials per phase (Inhale, Exhale)
- Each trial: 8 seconds (4s breathe + 4s tactile only)

Stereo channel layout:
- LEFT channel: Silence → Tactile trigger
- RIGHT channel: Breathing instruction → Silence (no looming)

========================================================================
"""

import numpy as np
import soundfile as sf
from pydub import AudioSegment
import os
from pathlib import Path

print("=" * 70)
print("BASELINE TRIALS GENERATOR (BREATHING + TACTILE)")
print("=" * 70)

# ========================================================================
# FILE PATHS
# ========================================================================

# Breathing instruction files (MP3 input)
INHALE_INSTRUCTION = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\0. BoxBreathingInstructions\Inhale-2-3-4-hold_v3_FIXED.mp3"
EXHALE_INSTRUCTION = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\0. BoxBreathingInstructions\Exhale-2-3-4-hold_FIXED.mp3"

# Input directory with 4-second baseline tactile stimuli
BASELINE_TACTILE_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\1. TactileStimuli\BaselineTactileTrials"

# Output directories for baseline trials
OUTPUT_DIR_INHALE = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\6. Baseline\Inhale"
OUTPUT_DIR_EXHALE = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\6. Baseline\Exhale"

# Baseline tactile files (4 seconds each)
BASELINE_TACTILE_FILES = [
    "baseline_tactile_stimulus_300ms.wav",
    "baseline_tactile_stimulus_800ms.wav",
    "baseline_tactile_stimulus_1500ms.wav",
    "baseline_tactile_stimulus_2200ms.wav",
    "baseline_tactile_stimulus_2700ms.wav",
]

# Design parameters
TARGET_SAMPLE_RATE = 44100  # Standard sample rate

# ========================================================================
# HELPER FUNCTIONS
# ========================================================================

def load_mp3_as_stereo_right(mp3_path, target_sr=44100):
    """
    Load MP3 file and convert to stereo with audio in RIGHT channel only
    
    Returns: numpy array (N, 2) and sample rate
    """
    print(f"  Loading MP3: {os.path.basename(mp3_path)}")
    
    # Check if file exists
    if not os.path.exists(mp3_path):
        raise FileNotFoundError(f"MP3 file not found: {mp3_path}")
    
    try:
        # Load MP3 using pydub
        audio = AudioSegment.from_mp3(mp3_path)
        
        # Convert to target sample rate
        if audio.frame_rate != target_sr:
            print(f"    Resampling from {audio.frame_rate} Hz to {target_sr} Hz...")
            audio = audio.set_frame_rate(target_sr)
        
        # Convert to mono if stereo (take first channel)
        if audio.channels > 1:
            audio = audio.set_channels(1)
        
        # Get samples as numpy array
        samples = np.array(audio.get_array_of_samples(), dtype=np.float32)
        
        # Normalize to [-1, 1]
        samples = samples / (2**15)
        
        # Create stereo array with audio in RIGHT channel only
        stereo_right = np.zeros((len(samples), 2), dtype=np.float32)
        stereo_right[:, 0] = 0.0        # LEFT channel = silence
        stereo_right[:, 1] = samples    # RIGHT channel = breathing instruction
        
        print(f"    Duration: {len(stereo_right) / target_sr:.2f}s")
        print(f"    Sample rate: {target_sr} Hz")
        print(f"    Channels: LEFT=silence, RIGHT=instruction")
        
        return stereo_right, target_sr
        
    except Exception as e:
        print(f"    ✗ Error reading MP3 file: {e}")
        print(f"    Full path: {mp3_path}")
        raise

def combine_breathing_and_baseline_tactile(breathing_audio, tactile_audio, sr, target_total_duration=8.0):
    """
    Combine breathing instruction with baseline tactile stimulus
    Adjusts tactile portion to ensure exactly target_total_duration
    
    Breathing audio: (N, 2) - LEFT=silence, RIGHT=instruction
    Tactile audio: (M, 2) - LEFT=tactile, RIGHT=silence (no looming)
    
    Result: (N+M, 2) - LEFT=tactile, RIGHT=instruction, exactly 8.0 seconds
    """
    # Calculate how long the tactile portion should be
    breathing_duration = len(breathing_audio) / sr
    tactile_duration_needed = target_total_duration - breathing_duration
    
    # Calculate required samples for tactile portion
    tactile_samples_needed = int(tactile_duration_needed * sr)
    current_tactile_samples = len(tactile_audio)
    
    # Adjust tactile audio length if needed
    if current_tactile_samples != tactile_samples_needed:
        # Need to pad or trim
        if current_tactile_samples < tactile_samples_needed:
            # Pad with silence at the end
            padding_needed = tactile_samples_needed - current_tactile_samples
            padding = np.zeros((padding_needed, 2))
            tactile_adjusted = np.vstack([tactile_audio, padding])
        else:
            # Trim (should not happen if input is correct, but handle anyway)
            tactile_adjusted = tactile_audio[:tactile_samples_needed]
    else:
        tactile_adjusted = tactile_audio
    
    # Ensure tactile audio has silence in RIGHT channel (no looming)
    tactile_baseline = tactile_adjusted.copy()
    tactile_baseline[:, 1] = 0.0  # Force RIGHT channel to silence (no looming)
    
    # Concatenate vertically (time dimension)
    combined = np.vstack([breathing_audio, tactile_baseline])
    
    # Verify exact duration
    actual_duration = len(combined) / sr
    
    return combined, actual_duration

# ========================================================================
# CREATE OUTPUT DIRECTORIES
# ========================================================================

os.makedirs(OUTPUT_DIR_INHALE, exist_ok=True)
os.makedirs(OUTPUT_DIR_EXHALE, exist_ok=True)

print(f"\nOutput directories:")
print(f"  Inhale: {OUTPUT_DIR_INHALE}")
print(f"  Exhale: {OUTPUT_DIR_EXHALE}")

# ========================================================================
# LOAD BREATHING INSTRUCTIONS
# ========================================================================

print("\n" + "-" * 70)
print("LOADING BREATHING INSTRUCTIONS")
print("-" * 70)

print("\n[1/2] Loading INHALE instruction...")
inhale_instruction, inhale_sr = load_mp3_as_stereo_right(INHALE_INSTRUCTION, TARGET_SAMPLE_RATE)

print("\n[2/2] Loading EXHALE instruction...")
exhale_instruction, exhale_sr = load_mp3_as_stereo_right(EXHALE_INSTRUCTION, TARGET_SAMPLE_RATE)

# ========================================================================
# PROCESS EACH BASELINE TACTILE STIMULUS
# ========================================================================

print("\n" + "=" * 70)
print("PROCESSING BASELINE TACTILE STIMULI")
print("=" * 70)

total_files = len(BASELINE_TACTILE_FILES)
success_count = 0
error_count = 0
errors = []

total_trials_created = 0

for idx, filename in enumerate(BASELINE_TACTILE_FILES, start=1):
    print(f"\n[{idx}/{total_files}] Processing: {filename}")
    
    try:
        # Full input path
        input_path = os.path.join(BASELINE_TACTILE_DIR, filename)
        
        # Check if file exists
        if not os.path.exists(input_path):
            print(f"  ✗ ERROR: File not found")
            error_count += 1
            errors.append(f"{filename} - File not found")
            continue
        
        # Load baseline tactile stimulus (should be 4 seconds, stereo)
        print(f"  Loading baseline tactile stimulus...")
        tactile_audio, tactile_sr = sf.read(input_path)
        
        # Verify it's stereo
        if len(tactile_audio.shape) != 2 or tactile_audio.shape[1] != 2:
            print(f"  ✗ ERROR: Not a stereo file")
            error_count += 1
            errors.append(f"{filename} - Not stereo")
            continue
        
        # Resample if needed
        if tactile_sr != TARGET_SAMPLE_RATE:
            print(f"  Resampling from {tactile_sr} Hz to {TARGET_SAMPLE_RATE} Hz...")
            ratio = TARGET_SAMPLE_RATE / tactile_sr
            new_length = int(len(tactile_audio) * ratio)
            tactile_audio = np.array([
                np.interp(
                    np.linspace(0, len(tactile_audio), new_length),
                    np.arange(len(tactile_audio)),
                    tactile_audio[:, ch]
                ) for ch in range(2)
            ]).T
            tactile_sr = TARGET_SAMPLE_RATE
        
        tactile_duration = len(tactile_audio) / tactile_sr
        print(f"    Duration: {tactile_duration:.3f} s")
        
        # Extract SOA info from filename (e.g., "300ms" from "baseline_tactile_stimulus_300ms.wav")
        soa_match = filename.split('_')[-1].replace('.wav', '')  # Gets "300ms"
        
        # Create INHALE baseline trial (single file)
        print(f"  Creating INHALE baseline trial...")
        inhale_combined, inhale_duration = combine_breathing_and_baseline_tactile(
            inhale_instruction,
            tactile_audio,
            TARGET_SAMPLE_RATE,
            target_total_duration=8.0
        )
        
        # Create descriptive filename
        output_filename = f"Baseline_Inhale_SOA_{soa_match}.wav"
        output_path = os.path.join(OUTPUT_DIR_INHALE, output_filename)
        sf.write(output_path, inhale_combined, TARGET_SAMPLE_RATE)
        
        print(f"    ✓ Saved: {output_filename} (duration: {inhale_duration:.3f}s)")
        total_trials_created += 1
        
        # Create EXHALE baseline trial (single file)
        print(f"  Creating EXHALE baseline trial...")
        exhale_combined, exhale_duration = combine_breathing_and_baseline_tactile(
            exhale_instruction,
            tactile_audio,
            TARGET_SAMPLE_RATE,
            target_total_duration=8.0
        )
        
        # Create descriptive filename
        output_filename = f"Baseline_Exhale_SOA_{soa_match}.wav"
        output_path = os.path.join(OUTPUT_DIR_EXHALE, output_filename)
        sf.write(output_path, exhale_combined, TARGET_SAMPLE_RATE)
        
        print(f"    ✓ Saved: {output_filename} (duration: {exhale_duration:.3f}s)")
        total_trials_created += 1
        
        success_count += 1
        
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        error_count += 1
        errors.append(f"{filename} - {str(e)}")
        import traceback
        traceback.print_exc()

# ========================================================================
# SUMMARY
# ========================================================================

print("\n" + "=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)

print(f"\nResults:")
print(f"  ✓ Successfully processed: {success_count}/{total_files} stimuli")
print(f"  ✗ Errors: {error_count}/{total_files} stimuli")

if error_count > 0:
    print(f"\nErrors encountered:")
    for error in errors:
        print(f"  • {error}")

print(f"\nBaseline trials created:")
print(f"  Total trials: {total_trials_created}")
print(f"  Breakdown:")
print(f"    • {success_count} SOAs")
print(f"    • 2 phases (Inhale, Exhale)")
print(f"    • {success_count} trials per phase")

print(f"\nOutput locations:")
print(f"  Inhale baseline: {OUTPUT_DIR_INHALE}")
print(f"  Exhale baseline: {OUTPUT_DIR_EXHALE}")

print(f"\nFile naming convention:")
print(f"  Baseline_Inhale_SOA_[delay].wav")
print(f"  Baseline_Exhale_SOA_[delay].wav")

print(f"\nTrial structure:")
print(f"  0.0 - ~4.04s: Breathing instruction (inhale/exhale)")
print(f"  ~4.04 - 8.00s: Breath hold with tactile only (no looming)")
print(f"  Total: EXACTLY 8.000 seconds per trial")

print(f"\nStereo channel layout:")
print(f"  LEFT:  Silence → Tactile trigger")
print(f"  RIGHT: Breathing instruction → Silence (NO looming audio)")

print(f"\nExperimental design:")
print(f"  ✓ 5 SOAs: 300ms, 800ms, 1500ms, 2200ms, 2700ms")
print(f"  ✓ Single file per SOA per phase")
print(f"  ✓ 5 baseline trials per phase")
print(f"  ✓ 10 baseline trial types total")
print(f"  ✓ EXACTLY 8.000s duration (matches audio-tactile trials)")

print("\n" + "=" * 70)

BASELINE TRIALS GENERATOR (BREATHING + TACTILE)

Output directories:
  Inhale: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\6. Baseline\Inhale
  Exhale: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\6. Baseline\Exhale

----------------------------------------------------------------------
LOADING BREATHING INSTRUCTIONS
----------------------------------------------------------------------

[1/2] Loading INHALE instruction...
  Loading MP3: Inhale-2-3-4-hold_v3_FIXED.mp3
    Duration: 4.04s
    Sample rate: 44100 Hz
    Channels: LEFT=silence, RIGHT=instruction

[2/2] Loading EXHALE instruction...
  Loading MP3: Exhale-2-3-4-hold_FIXED.mp3
    Duration: 4.04s
    Sample rate: 44100 Hz
    Channels: LEFT=silence, RIGHT=instruction

PROCESSING BASELINE TACTILE STIMULI

[1/5] Processing: baseline_tactile_stimulus_300ms.wav
  Loading baseline tactile stimulus...
    Duration: 4.000 s
  Creating INHALE baseline trial...
    ✓ Saved: Baseline_Inhale_SOA_

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_33120\4030394230.py", line 216, in <module>
    tactile_audio, tactile_sr = sf.read(input_path)
                                ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\envs\PPS\Lib\site-packages\soundfile.py", line 308, in read
    data = f.read(frames, dtype, always_2d, fill_value, out)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\envs\PPS\Lib\site-packages\soundfile.py", line 942, in read
    frames = self._array_io('read', out, frames)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\envs\PPS\Lib\site-packages\soundfile.py", line 1394, in _array_io
    return self._cdata_io(action, cdata, ctype, frames)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\envs\PPS\Lib\site-packages\soundfile.py", line 1404, in _cdata_io
    _error_che

Generate Catch Trials

========================================================================
7. Create Catch Trials
========================================================================

In [43]:
"""
========================================================================
CATCH TRIALS GENERATOR (BREATHING + LOOMING AUDIO ONLY)
========================================================================

This script creates catch trials by combining breathing instructions
with looming audio stimuli ONLY (no tactile stimulation).

Purpose: Monitor attention and prevent response habituation by including
trials where participants should withhold their response (no tactile).

Process:
1. Load breathing instructions (Inhale/Exhale) - ~4s each
2. Load looming audio stimuli (4 noise types)
3. Extend looming audio from 3000ms to exactly required length
4. Combine to create 8-second catch trials
5. Create single file per SOA per noise per phase
6. Save to Inhale and Exhale catch folders

Design:
- 5 SOAs × 4 noise types × 2 phases = 40 catch trial types
- Noise type varies across SOAs
- One file per unique combination

Stereo channel layout:
- LEFT channel: Silence (no tactile)
- RIGHT channel: Breathing instruction → Looming audio

========================================================================
"""

import numpy as np
import soundfile as sf
from pydub import AudioSegment
import os
from pathlib import Path

print("=" * 70)
print("CATCH TRIALS GENERATOR (BREATHING + LOOMING ONLY)")
print("=" * 70)

# ========================================================================
# FILE PATHS
# ========================================================================

# Breathing instruction files (MP3 input)
INHALE_INSTRUCTION = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\0. BoxBreathingInstructions\Inhale-2-3-4-hold_v3_FIXED.mp3"
EXHALE_INSTRUCTION = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\0. BoxBreathingInstructions\Exhale-2-3-4-hold_FIXED.mp3"

# Input directory with looming audio stimuli
# These are the original 3000ms looming audio files
LOOMING_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\2. LoomingStimuli"

# Output directories for catch trials
OUTPUT_DIR_INHALE = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\7. Catch\Inhale"
OUTPUT_DIR_EXHALE = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\7. Catch\Exhale"

# Looming audio files (3000ms each)
LOOMING_FILES = {
    'pink': 'Loom-1-pink.wav',
    'blue': 'Loom-2-blue.wav',
    'white': 'Loom-3-white.wav',
    'brown': 'Loom-4-brown.wav',
}

# SOA values
SOAS = [300, 800, 1500, 2200, 2700]

TARGET_SAMPLE_RATE = 44100  # Standard sample rate
TARGET_TOTAL_DURATION = 8.0 # Target: 8 seconds per trial

# ========================================================================
# HELPER FUNCTIONS
# ========================================================================

def load_mp3_as_stereo_right(mp3_path, target_sr=44100):
    """
    Load MP3 file and convert to stereo with audio in RIGHT channel only
    
    Returns: numpy array (N, 2) and sample rate
    """
    print(f"  Loading MP3: {os.path.basename(mp3_path)}")
    
    # Check if file exists
    if not os.path.exists(mp3_path):
        raise FileNotFoundError(f"MP3 file not found: {mp3_path}")
    
    try:
        # Load MP3 using pydub
        audio = AudioSegment.from_mp3(mp3_path)
        
        # Convert to target sample rate
        if audio.frame_rate != target_sr:
            print(f"    Resampling from {audio.frame_rate} Hz to {target_sr} Hz...")
            audio = audio.set_frame_rate(target_sr)
        
        # Convert to mono if stereo (take first channel)
        if audio.channels > 1:
            audio = audio.set_channels(1)
        
        # Get samples as numpy array
        samples = np.array(audio.get_array_of_samples(), dtype=np.float32)
        
        # Normalize to [-1, 1]
        samples = samples / (2**15)
        
        # Create stereo array with audio in RIGHT channel only
        stereo_right = np.zeros((len(samples), 2), dtype=np.float32)
        stereo_right[:, 0] = 0.0        # LEFT channel = silence
        stereo_right[:, 1] = samples    # RIGHT channel = breathing instruction
        
        print(f"    Duration: {len(stereo_right) / target_sr:.2f}s")
        print(f"    Sample rate: {target_sr} Hz")
        print(f"    Channels: LEFT=silence, RIGHT=instruction")
        
        return stereo_right, target_sr
        
    except Exception as e:
        print(f"    ✗ Error reading MP3 file: {e}")
        print(f"    Full path: {mp3_path}")
        raise

def extend_looming_to_exact_duration(looming_audio, sr, breathing_duration, target_total_duration=8.0):
    """
    Extend looming audio to create exactly target_total_duration after combining with breathing
    Similar to looming-tactile approach but for audio-only
    
    Returns extended stereo audio with looming in RIGHT channel, silence in LEFT
    """
    # Calculate how long the looming portion should be
    looming_duration_needed = target_total_duration - breathing_duration
    
    # Calculate required samples
    looming_samples_needed = int(looming_duration_needed * sr)
    current_samples = len(looming_audio)
    
    # Calculate padding needed
    total_padding_needed = looming_samples_needed - current_samples
    
    # Split padding evenly before and after
    pad_start_samples = total_padding_needed // 2
    pad_end_samples = total_padding_needed - pad_start_samples
    
    # If looming audio is mono, convert to stereo
    if len(looming_audio.shape) == 1:
        looming_audio = np.column_stack([looming_audio, looming_audio])
    
    # Create extended audio with looming in RIGHT channel only, LEFT is silent
    extended = np.zeros((looming_samples_needed, 2))
    
    # Add padding before
    start_idx = pad_start_samples
    end_idx = start_idx + current_samples
    
    # Put looming audio in RIGHT channel only
    extended[start_idx:end_idx, 0] = 0.0  # LEFT = silence (no tactile)
    extended[start_idx:end_idx, 1] = looming_audio[:, 0]  # RIGHT = looming audio
    
    actual_duration = len(extended) / sr
    
    return extended, actual_duration

def combine_breathing_and_catch_looming(breathing_audio, looming_audio, sr):
    """
    Combine breathing instruction with looming audio for catch trials
    
    Breathing audio: (N, 2) - LEFT=silence, RIGHT=instruction
    Looming audio: (M, 2) - LEFT=silence (no tactile), RIGHT=looming
    
    Result: (N+M, 2) - LEFT=silence, RIGHT=instruction+looming
    """
    # Concatenate vertically (time dimension)
    combined = np.vstack([breathing_audio, looming_audio])
    
    # Verify duration
    actual_duration = len(combined) / sr
    
    return combined, actual_duration

# ========================================================================
# CREATE OUTPUT DIRECTORIES
# ========================================================================

os.makedirs(OUTPUT_DIR_INHALE, exist_ok=True)
os.makedirs(OUTPUT_DIR_EXHALE, exist_ok=True)

print(f"\nOutput directories:")
print(f"  Inhale: {OUTPUT_DIR_INHALE}")
print(f"  Exhale: {OUTPUT_DIR_EXHALE}")

# ========================================================================
# LOAD BREATHING INSTRUCTIONS
# ========================================================================

print("\n" + "-" * 70)
print("LOADING BREATHING INSTRUCTIONS")
print("-" * 70)

print("\n[1/2] Loading INHALE instruction...")
inhale_instruction, inhale_sr = load_mp3_as_stereo_right(INHALE_INSTRUCTION, TARGET_SAMPLE_RATE)
breathing_inhale_duration = len(inhale_instruction) / TARGET_SAMPLE_RATE

print("\n[2/2] Loading EXHALE instruction...")
exhale_instruction, exhale_sr = load_mp3_as_stereo_right(EXHALE_INSTRUCTION, TARGET_SAMPLE_RATE)
breathing_exhale_duration = len(exhale_instruction) / TARGET_SAMPLE_RATE

# ========================================================================
# LOAD LOOMING AUDIO STIMULI
# ========================================================================

print("\n" + "-" * 70)
print("LOADING LOOMING AUDIO STIMULI")
print("-" * 70)

looming_audio_dict = {}

for noise_type, filename in LOOMING_FILES.items():
    print(f"\nLoading {noise_type} noise...")
    looming_path = os.path.join(LOOMING_DIR, filename)
    
    if not os.path.exists(looming_path):
        print(f"  ✗ ERROR: File not found: {filename}")
        continue
    
    looming_audio, looming_sr = sf.read(looming_path)
    
    # Resample if needed
    if looming_sr != TARGET_SAMPLE_RATE:
        print(f"  Resampling from {looming_sr} Hz to {TARGET_SAMPLE_RATE} Hz...")
        from scipy import signal as sp_signal
        num_samples = int(len(looming_audio) * TARGET_SAMPLE_RATE / looming_sr)
        if len(looming_audio.shape) == 1:
            looming_audio = sp_signal.resample(looming_audio, num_samples)
        else:
            looming_audio = np.array([
                sp_signal.resample(looming_audio[:, ch], num_samples)
                for ch in range(looming_audio.shape[1])
            ]).T
    
    looming_audio_dict[noise_type] = looming_audio
    print(f"  ✓ Loaded {noise_type} noise ({len(looming_audio) / TARGET_SAMPLE_RATE * 1000:.0f} ms)")

# ========================================================================
# GENERATE CATCH TRIALS
# ========================================================================

print("\n" + "=" * 70)
print("GENERATING CATCH TRIALS")
print("=" * 70)

# Check if we have looming audio loaded
if not looming_audio_dict:
    print("\n✗ ERROR: No looming audio files were loaded!")
    print("Please check that the looming audio files exist in:")
    print(f"  {LOOMING_DIR}")
    print("\nExpected files:")
    for noise_type, filename in LOOMING_FILES.items():
        print(f"  • {filename}")
    print("\nExiting...")
    exit(1)

total_trials_created = 0

# Generate one catch trial per SOA × Noise × Phase combination
for soa in SOAS:
    print(f"\n[SOA: {soa}ms]")
    
    for noise_type, looming_audio in looming_audio_dict.items():
        print(f"  Processing {noise_type} noise...")
        
        # === INHALE CATCH TRIAL ===
        # Extend looming audio to exact duration
        extended_looming_inhale, _ = extend_looming_to_exact_duration(
            looming_audio,
            TARGET_SAMPLE_RATE,
            breathing_inhale_duration,
            TARGET_TOTAL_DURATION
        )
        
        # Combine with inhale instruction
        inhale_combined, inhale_duration = combine_breathing_and_catch_looming(
            inhale_instruction,
            extended_looming_inhale,
            TARGET_SAMPLE_RATE
        )
        
        # Save inhale catch trial
        output_filename = f"Catch_Inhale_SOA_{soa}ms_{noise_type}.wav"
        output_path = os.path.join(OUTPUT_DIR_INHALE, output_filename)
        sf.write(output_path, inhale_combined, TARGET_SAMPLE_RATE)
        
        print(f"    ✓ Inhale: {output_filename} ({inhale_duration:.3f}s)")
        total_trials_created += 1
        
        # === EXHALE CATCH TRIAL ===
        # Extend looming audio to exact duration
        extended_looming_exhale, _ = extend_looming_to_exact_duration(
            looming_audio,
            TARGET_SAMPLE_RATE,
            breathing_exhale_duration,
            TARGET_TOTAL_DURATION
        )
        
        # Combine with exhale instruction
        exhale_combined, exhale_duration = combine_breathing_and_catch_looming(
            exhale_instruction,
            extended_looming_exhale,
            TARGET_SAMPLE_RATE
        )
        
        # Save exhale catch trial
        output_filename = f"Catch_Exhale_SOA_{soa}ms_{noise_type}.wav"
        output_path = os.path.join(OUTPUT_DIR_EXHALE, output_filename)
        sf.write(output_path, exhale_combined, TARGET_SAMPLE_RATE)
        
        print(f"    ✓ Exhale: {output_filename} ({exhale_duration:.3f}s)")
        total_trials_created += 1

# ========================================================================
# SUMMARY
# ========================================================================

print("\n" + "=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)

print(f"\nCatch trials created:")
print(f"  Total trials: {total_trials_created}")
print(f"  Breakdown:")
print(f"    • 5 SOAs: {SOAS}")
print(f"    • 4 noise types: {list(looming_audio_dict.keys())}")
print(f"    • 2 phases (Inhale, Exhale)")
print(f"    • {total_trials_created // 2} trials per phase")

print(f"\nOutput locations:")
print(f"  Inhale catch: {OUTPUT_DIR_INHALE}")
print(f"  Exhale catch: {OUTPUT_DIR_EXHALE}")

print(f"\nFile naming convention:")
print(f"  Catch_Inhale_SOA_[delay]_[noise].wav")
print(f"  Catch_Exhale_SOA_[delay]_[noise].wav")

print(f"\nTrial structure:")
print(f"  0.0 - ~4.04s: Breathing instruction (inhale/exhale)")
print(f"  ~4.04 - 8.00s: Breath hold with looming audio ONLY")
print(f"  Total: EXACTLY 8.000 seconds per trial")

print(f"\nStereo channel layout:")
print(f"  LEFT:  Silence (NO tactile trigger)")
print(f"  RIGHT: Breathing instruction → Looming audio")

print(f"\nExperimental design:")
print(f"  ✓ 5 SOAs × 4 noise types × 2 phases = {total_trials_created} catch trial types")
print(f"  ✓ Single file per unique combination")
print(f"  ✓ EXACTLY 8.000s duration")

print(f"\nPurpose:")
print(f"  Catch trials ensure participants:")
print(f"  • Maintain attention throughout experiment")
print(f"  • Respond only to tactile stimuli (not audio)")
print(f"  • Prevent automatic/habitual responses")
print(f"  • Correct rejection = withhold response (no tactile present)")

print("\n" + "=" * 70)

CATCH TRIALS GENERATOR (BREATHING + LOOMING ONLY)

Output directories:
  Inhale: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\7. Catch\Inhale
  Exhale: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\7. Catch\Exhale

----------------------------------------------------------------------
LOADING BREATHING INSTRUCTIONS
----------------------------------------------------------------------

[1/2] Loading INHALE instruction...
  Loading MP3: Inhale-2-3-4-hold_v3_FIXED.mp3
    Duration: 4.04s
    Sample rate: 44100 Hz
    Channels: LEFT=silence, RIGHT=instruction

[2/2] Loading EXHALE instruction...
  Loading MP3: Exhale-2-3-4-hold_FIXED.mp3
    Duration: 4.04s
    Sample rate: 44100 Hz
    Channels: LEFT=silence, RIGHT=instruction

----------------------------------------------------------------------
LOADING LOOMING AUDIO STIMULI
----------------------------------------------------------------------

Loading pink noise...
  ✓ Loaded pink noise (3000 ms

========================================================================
8. Create Master Blocks CSV (Template for randomization)
========================================================================

In [47]:
import pandas as pd
from pathlib import Path

# Set base directory
base_dir = Path(r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli")

# Create output directory
output_dir = base_dir / "8.Master_Blocks"
output_dir.mkdir(exist_ok=True)

print("="*80)
print("MASTER BLOCK GENERATION - MANUALLY DEFINED")
print("="*80)

# ============================================================================
# MASTER BLOCK 1 - MANUALLY DEFINED
# ============================================================================

# Each row: [Trial_Type, SOA_ms, Noise_Type, Respiratory_Phase]
master_block_1_data = [
    # Audio-Tactile trials (20 trials: 10 Inhale, 10 Exhale)
    ['Audio-Tactile', 300, 'Pink', 'Inhale'],
    ['Audio-Tactile', 300, 'Blue', 'Exhale'],
    ['Audio-Tactile', 300, 'White', 'Inhale'],
    ['Audio-Tactile', 300, 'Brown', 'Exhale'],
    ['Audio-Tactile', 800, 'Pink', 'Inhale'],
    ['Audio-Tactile', 800, 'Blue', 'Exhale'],
    ['Audio-Tactile', 800, 'White', 'Inhale'],
    ['Audio-Tactile', 800, 'Brown', 'Exhale'],
    ['Audio-Tactile', 1500, 'Pink', 'Inhale'],
    ['Audio-Tactile', 1500, 'Blue', 'Exhale'],
    ['Audio-Tactile', 1500, 'White', 'Inhale'],
    ['Audio-Tactile', 1500, 'Brown', 'Exhale'],
    ['Audio-Tactile', 2200, 'Pink', 'Inhale'],
    ['Audio-Tactile', 2200, 'Blue', 'Exhale'],
    ['Audio-Tactile', 2200, 'White', 'Inhale'],
    ['Audio-Tactile', 2200, 'Brown', 'Exhale'],
    ['Audio-Tactile', 2700, 'Pink', 'Inhale'],
    ['Audio-Tactile', 2700, 'Blue', 'Exhale'],
    ['Audio-Tactile', 2700, 'White', 'Inhale'],
    ['Audio-Tactile', 2700, 'Brown', 'Exhale'],
    
    # Baseline trials (20 trials: 10 Inhale, 10 Exhale)
    ['Baseline', 300, 'N/A', 'Inhale'],
    ['Baseline', 300, 'N/A', 'Exhale'],
    ['Baseline', 300, 'N/A', 'Inhale'],
    ['Baseline', 300, 'N/A', 'Exhale'],
    ['Baseline', 800, 'N/A', 'Inhale'],
    ['Baseline', 800, 'N/A', 'Exhale'],
    ['Baseline', 800, 'N/A', 'Inhale'],
    ['Baseline', 800, 'N/A', 'Exhale'],
    ['Baseline', 1500, 'N/A', 'Inhale'],
    ['Baseline', 1500, 'N/A', 'Exhale'],
    ['Baseline', 1500, 'N/A', 'Inhale'],
    ['Baseline', 1500, 'N/A', 'Exhale'],
    ['Baseline', 2200, 'N/A', 'Inhale'],
    ['Baseline', 2200, 'N/A', 'Exhale'],
    ['Baseline', 2200, 'N/A', 'Inhale'],
    ['Baseline', 2200, 'N/A', 'Exhale'],
    ['Baseline', 2700, 'N/A', 'Inhale'],
    ['Baseline', 2700, 'N/A', 'Exhale'],
    ['Baseline', 2700, 'N/A', 'Inhale'],
    ['Baseline', 2700, 'N/A', 'Exhale'],
    
    # Catch trials (4 trials: 2 Inhale, 2 Exhale)
    ['Catch', 800, 'Pink', 'Inhale'],
    ['Catch', 1500, 'Brown', 'Exhale'],
    ['Catch', 2200, 'White', 'Inhale'],
    ['Catch', 2700, 'Blue', 'Exhale'],
]

# ============================================================================
# MASTER BLOCK 2 - MANUALLY DEFINED (Complementary to Block 1)
# ============================================================================

master_block_2_data = [
    # Audio-Tactile trials (20 trials: 10 Inhale, 10 Exhale)
    # These are the complementary combinations NOT in Block 1
    ['Audio-Tactile', 300, 'Pink', 'Exhale'],
    ['Audio-Tactile', 300, 'Blue', 'Inhale'],
    ['Audio-Tactile', 300, 'White', 'Exhale'],
    ['Audio-Tactile', 300, 'Brown', 'Inhale'],
    ['Audio-Tactile', 800, 'Pink', 'Exhale'],
    ['Audio-Tactile', 800, 'Blue', 'Inhale'],
    ['Audio-Tactile', 800, 'White', 'Exhale'],
    ['Audio-Tactile', 800, 'Brown', 'Inhale'],
    ['Audio-Tactile', 1500, 'Pink', 'Exhale'],
    ['Audio-Tactile', 1500, 'Blue', 'Inhale'],
    ['Audio-Tactile', 1500, 'White', 'Exhale'],
    ['Audio-Tactile', 1500, 'Brown', 'Inhale'],
    ['Audio-Tactile', 2200, 'Pink', 'Exhale'],
    ['Audio-Tactile', 2200, 'Blue', 'Inhale'],
    ['Audio-Tactile', 2200, 'White', 'Exhale'],
    ['Audio-Tactile', 2200, 'Brown', 'Inhale'],
    ['Audio-Tactile', 2700, 'Pink', 'Exhale'],
    ['Audio-Tactile', 2700, 'Blue', 'Inhale'],
    ['Audio-Tactile', 2700, 'White', 'Exhale'],
    ['Audio-Tactile', 2700, 'Brown', 'Inhale'],
    
    # Baseline trials (20 trials: 10 Inhale, 10 Exhale) - Same as Block 1
    ['Baseline', 300, 'N/A', 'Inhale'],
    ['Baseline', 300, 'N/A', 'Exhale'],
    ['Baseline', 300, 'N/A', 'Inhale'],
    ['Baseline', 300, 'N/A', 'Exhale'],
    ['Baseline', 800, 'N/A', 'Inhale'],
    ['Baseline', 800, 'N/A', 'Exhale'],
    ['Baseline', 800, 'N/A', 'Inhale'],
    ['Baseline', 800, 'N/A', 'Exhale'],
    ['Baseline', 1500, 'N/A', 'Inhale'],
    ['Baseline', 1500, 'N/A', 'Exhale'],
    ['Baseline', 1500, 'N/A', 'Inhale'],
    ['Baseline', 1500, 'N/A', 'Exhale'],
    ['Baseline', 2200, 'N/A', 'Inhale'],
    ['Baseline', 2200, 'N/A', 'Exhale'],
    ['Baseline', 2200, 'N/A', 'Inhale'],
    ['Baseline', 2200, 'N/A', 'Exhale'],
    ['Baseline', 2700, 'N/A', 'Inhale'],
    ['Baseline', 2700, 'N/A', 'Exhale'],
    ['Baseline', 2700, 'N/A', 'Inhale'],
    ['Baseline', 2700, 'N/A', 'Exhale'],
    
    # Catch trials (4 trials: 2 Inhale, 2 Exhale) - Different from Block 1
    ['Catch', 300, 'Blue', 'Inhale'],
    ['Catch', 800, 'Pink', 'Exhale'],
    ['Catch', 1500, 'Brown', 'Inhale'],
    ['Catch', 2700, 'White', 'Exhale'],
]

# ============================================================================
# CREATE DATAFRAMES
# ============================================================================

# Create Block 1 DataFrame
df_block1 = pd.DataFrame(master_block_1_data, columns=['Trial_Type', 'SOA_ms', 'Noise_Type', 'Respiratory_Phase'])
df_block1.insert(0, 'Trial_Number', range(1, len(df_block1) + 1))

# Create Block 2 DataFrame
df_block2 = pd.DataFrame(master_block_2_data, columns=['Trial_Type', 'SOA_ms', 'Noise_Type', 'Respiratory_Phase'])
df_block2.insert(0, 'Trial_Number', range(1, len(df_block2) + 1))

# ============================================================================
# VERIFICATION
# ============================================================================

print("\n" + "="*80)
print("MASTER BLOCK 1 VERIFICATION")
print("="*80)

# Count by trial type
print("\nTrial type counts:")
print(df_block1['Trial_Type'].value_counts().to_string())

# Count by phase
inhale_count_1 = (df_block1['Respiratory_Phase'] == 'Inhale').sum()
exhale_count_1 = (df_block1['Respiratory_Phase'] == 'Exhale').sum()
print(f"\nRespiratory phase balance:")
print(f"  Inhale: {inhale_count_1}")
print(f"  Exhale: {exhale_count_1}")
if inhale_count_1 == exhale_count_1:
    print(f"  ✓ PERFECTLY BALANCED")
else:
    print(f"  ✗ IMBALANCED (diff: {abs(inhale_count_1 - exhale_count_1)})")

# Check alternation
print(f"\nChecking Inhale-Exhale alternation:")
alternation_breaks = 0
for i in range(len(df_block1) - 1):
    if df_block1.iloc[i]['Respiratory_Phase'] == df_block1.iloc[i+1]['Respiratory_Phase']:
        alternation_breaks += 1

print(f"  Perfect alternation: {alternation_breaks == 0}")
print(f"  Total trials: {len(df_block1)}")

# Display sample
print("\nFirst 10 rows:")
print(df_block1.head(10).to_string(index=False))
print("\nLast 5 rows:")
print(df_block1.tail(5).to_string(index=False))

print("\n" + "="*80)
print("MASTER BLOCK 2 VERIFICATION")
print("="*80)

# Count by trial type
print("\nTrial type counts:")
print(df_block2['Trial_Type'].value_counts().to_string())

# Count by phase
inhale_count_2 = (df_block2['Respiratory_Phase'] == 'Inhale').sum()
exhale_count_2 = (df_block2['Respiratory_Phase'] == 'Exhale').sum()
print(f"\nRespiratory phase balance:")
print(f"  Inhale: {inhale_count_2}")
print(f"  Exhale: {exhale_count_2}")
if inhale_count_2 == exhale_count_2:
    print(f"  ✓ PERFECTLY BALANCED")
else:
    print(f"  ✗ IMBALANCED (diff: {abs(inhale_count_2 - exhale_count_2)})")

# Check alternation
print(f"\nChecking Inhale-Exhale alternation:")
alternation_breaks = 0
for i in range(len(df_block2) - 1):
    if df_block2.iloc[i]['Respiratory_Phase'] == df_block2.iloc[i+1]['Respiratory_Phase']:
        alternation_breaks += 1

print(f"  Perfect alternation: {alternation_breaks == 0}")
print(f"  Total trials: {len(df_block2)}")

# Display sample
print("\nFirst 10 rows:")
print(df_block2.head(10).to_string(index=False))
print("\nLast 5 rows:")
print(df_block2.tail(5).to_string(index=False))

# ============================================================================
# VERIFY COMPLEMENTARITY
# ============================================================================

print("\n" + "="*80)
print("COMPLEMENTARITY VERIFICATION")
print("="*80)

# Extract audio-tactile combinations
block1_at = set()
for _, row in df_block1[df_block1['Trial_Type'] == 'Audio-Tactile'].iterrows():
    block1_at.add((row['SOA_ms'], row['Noise_Type'], row['Respiratory_Phase']))

block2_at = set()
for _, row in df_block2[df_block2['Trial_Type'] == 'Audio-Tactile'].iterrows():
    block2_at.add((row['SOA_ms'], row['Noise_Type'], row['Respiratory_Phase']))

overlap = block1_at & block2_at
union = block1_at | block2_at

print(f"\nAudio-tactile combinations:")
print(f"  Block 1: {len(block1_at)} unique combinations")
print(f"  Block 2: {len(block2_at)} unique combinations")
print(f"  Overlap: {len(overlap)} (should be 0)")
print(f"  Union: {len(union)} (should be 40)")

if len(overlap) == 0 and len(union) == 40:
    print(f"  ✓ PERFECT COMPLEMENTARITY")
else:
    print(f"  ✗ COMPLEMENTARITY ERROR")

# ============================================================================
# SAVE FILES
# ============================================================================

print("\n" + "="*80)
print("SAVING FILES")
print("="*80)

block1_path = output_dir / "Master_Block_1.csv"
df_block1.to_csv(block1_path, index=False)
print(f"✓ Saved: {block1_path}")

block2_path = output_dir / "Master_Block_2.csv"
df_block2.to_csv(block2_path, index=False)
print(f"✓ Saved: {block2_path}")

print("\n" + "="*80)
print("✓ MASTER BLOCKS GENERATED SUCCESSFULLY")
print("="*80)
print(f"\nSummary:")
print(f"  • Both blocks have exactly 22 Inhale and 22 Exhale trials")
print(f"  • Perfect I-E-I-E alternation throughout")
print(f"  • Audio-tactile trials are perfectly complementary")
print(f"  • Ready to be randomized into blocks A-F")
print("="*80)

MASTER BLOCK GENERATION - MANUALLY DEFINED

MASTER BLOCK 1 VERIFICATION

Trial type counts:
Trial_Type
Audio-Tactile    20
Baseline         20
Catch             4

Respiratory phase balance:
  Inhale: 22
  Exhale: 22
  ✓ PERFECTLY BALANCED

Checking Inhale-Exhale alternation:
  Perfect alternation: True
  Total trials: 44

First 10 rows:
 Trial_Number    Trial_Type  SOA_ms Noise_Type Respiratory_Phase
            1 Audio-Tactile     300       Pink            Inhale
            2 Audio-Tactile     300       Blue            Exhale
            3 Audio-Tactile     300      White            Inhale
            4 Audio-Tactile     300      Brown            Exhale
            5 Audio-Tactile     800       Pink            Inhale
            6 Audio-Tactile     800       Blue            Exhale
            7 Audio-Tactile     800      White            Inhale
            8 Audio-Tactile     800      Brown            Exhale
            9 Audio-Tactile    1500       Pink            Inhale
          

========================================================================
BLOCK-BASED COUNTERBALANCING SEQUENCE GENERATOR
========================================================================

In [48]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

# Set base directory
base_dir = Path(r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli")
master_blocks_dir = base_dir / "8.Master_Blocks"
output_dir = base_dir / "9.Experiment_Blocks"
output_dir.mkdir(exist_ok=True)

# Audio file directories
audio_dir_inhale = base_dir / "5. Breathingphase+LoomingXTactile" / "Inhale"
audio_dir_exhale = base_dir / "5. Breathingphase+LoomingXTactile" / "Exhale"
baseline_dir_inhale = base_dir / "6. Baseline" / "Inhale"
baseline_dir_exhale = base_dir / "6. Baseline" / "Exhale"
catch_dir_inhale = base_dir / "7. Catch" / "Inhale"
catch_dir_exhale = base_dir / "7. Catch" / "Exhale"

print("="*80)
print("EXPERIMENT BLOCKS GENERATOR")
print("="*80)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def check_max_runs(sequence, feature_col, max_run=3):
    """Check if any feature has more than max_run consecutive occurrences."""
    current_val = None
    current_run = 0
    
    for val in sequence[feature_col]:
        if val == current_val:
            current_run += 1
            if current_run > max_run:
                return False
        else:
            current_val = val
            current_run = 1
    return True

def check_catch_spacing(sequence, min_spacing=8):
    """Ensure catch trials are not too close together."""
    catch_indices = sequence[sequence['Trial_Type'] == 'Catch'].index.tolist()
    
    if len(catch_indices) < 2:
        return True
    
    for i in range(len(catch_indices) - 1):
        if catch_indices[i+1] - catch_indices[i] < min_spacing:
            return False
    return True

def check_temporal_balance(sequence, n_segments=4):
    """Check that trial types are balanced across temporal segments."""
    segment_size = len(sequence) // n_segments
    trial_types = ['Audio-Tactile', 'Baseline']
    
    for trial_type in trial_types:
        counts_per_segment = []
        for seg in range(n_segments):
            start_idx = seg * segment_size
            end_idx = (seg + 1) * segment_size if seg < n_segments - 1 else len(sequence)
            segment_data = sequence.iloc[start_idx:end_idx]
            count = (segment_data['Trial_Type'] == trial_type).sum()
            counts_per_segment.append(count)
        
        mean_count = np.mean(counts_per_segment)
        if mean_count > 0:
            cv = np.std(counts_per_segment) / mean_count
            if cv > 0.5:
                return False
    
    return True

def constrained_randomization(trials_df, max_attempts=1000, seed=None):
    """
    Randomize trial order with constraints while maintaining I-E-I-E alternation.
    """
    if seed is not None:
        np.random.seed(seed)
    
    for attempt in range(max_attempts):
        # Separate by phase
        inhale_trials = trials_df[trials_df['Respiratory_Phase'] == 'Inhale'].copy()
        exhale_trials = trials_df[trials_df['Respiratory_Phase'] == 'Exhale'].copy()
        
        # Shuffle independently
        inhale_trials = inhale_trials.sample(frac=1).reset_index(drop=True)
        exhale_trials = exhale_trials.sample(frac=1).reset_index(drop=True)
        
        # Interleave to maintain phase alternation
        randomized = []
        for i in range(max(len(inhale_trials), len(exhale_trials))):
            if i < len(inhale_trials):
                randomized.append(inhale_trials.iloc[i])
            if i < len(exhale_trials):
                randomized.append(exhale_trials.iloc[i])
        
        result = pd.DataFrame(randomized).reset_index(drop=True)
        
        # Apply constraints
        if not check_max_runs(result, 'Trial_Type', max_run=3):
            continue
        
        if not check_catch_spacing(result, min_spacing=8):
            continue
        
        if not check_temporal_balance(result, n_segments=4):
            continue
        
        print(f"  ✓ Valid sequence found in {attempt + 1} attempts")
        return result
    
    print(f"  ⚠ WARNING: Could not satisfy all constraints in {max_attempts} attempts")
    return result

def get_audio_filename(row):
    """
    Generate the expected audio filename based on trial parameters.
    """
    trial_type = row['Trial_Type']
    soa = row['SOA_ms']
    noise = row['Noise_Type']
    phase = row['Respiratory_Phase']
    
    if trial_type == 'Audio-Tactile':
        # Audio-Tactile file format: Stereo_LEFT_tactile_stimulus_300ms__RIGHT_Loom-1-pink.wav
        noise_map = {'Pink': '1-pink', 'Blue': '2-blue', 'White': '3-white', 'Brown': '4-brown'}
        noise_code = noise_map[noise]
        filename = f"Stereo_LEFT_tactile_stimulus_{soa}ms__RIGHT_Loom-{noise_code}.wav"
        
    elif trial_type == 'Baseline':
        # Baseline file format: Baseline_Inhale_SOA_300ms.wav
        filename = f"Baseline_{phase}_SOA_{soa}ms.wav"
        
    elif trial_type == 'Catch':
        # Catch file format: Catch_Inhale_SOA_300ms_pink.wav
        noise_lower = noise.lower()
        filename = f"Catch_{phase}_SOA_{soa}ms_{noise_lower}.wav"
    
    return filename

def get_audio_filepath(row):
    """
    Get the full path to the audio file.
    """
    filename = get_audio_filename(row)
    trial_type = row['Trial_Type']
    phase = row['Respiratory_Phase']
    
    if trial_type == 'Audio-Tactile':
        if phase == 'Inhale':
            return audio_dir_inhale / filename
        else:
            return audio_dir_exhale / filename
            
    elif trial_type == 'Baseline':
        if phase == 'Inhale':
            return baseline_dir_inhale / filename
        else:
            return baseline_dir_exhale / filename
            
    elif trial_type == 'Catch':
        if phase == 'Inhale':
            return catch_dir_inhale / filename
        else:
            return catch_dir_exhale / filename
    
    return None

def verify_audio_files(df, block_name):
    """
    Verify that all required audio files exist.
    """
    print(f"\n  Verifying audio files for {block_name}...")
    missing_files = []
    
    for idx, row in df.iterrows():
        filepath = get_audio_filepath(row)
        if filepath is None or not filepath.exists():
            missing_files.append({
                'trial': idx + 1,
                'type': row['Trial_Type'],
                'expected': filepath
            })
    
    if len(missing_files) == 0:
        print(f"  ✓ All {len(df)} audio files found")
        return True
    else:
        print(f"  ✗ {len(missing_files)} audio files missing:")
        for mf in missing_files[:5]:  # Show first 5
            print(f"    Trial {mf['trial']}: {mf['expected']}")
        return False

def create_wav_file_list(df, block_name, output_dir):
    """
    Create a text file with the ordered list of WAV files for playback.
    """
    wav_list_path = output_dir / f"{block_name}_wav_playlist.txt"
    
    with open(wav_list_path, 'w') as f:
        f.write(f"# WAV Playlist for {block_name}\n")
        f.write(f"# Total trials: {len(df)}\n")
        f.write(f"# Generated automatically\n\n")
        
        for idx, row in df.iterrows():
            filepath = get_audio_filepath(row)
            if filepath:
                # Write relative path from output directory
                rel_path = filepath.relative_to(base_dir)
                f.write(f"{rel_path}\n")
    
    print(f"  ✓ Created WAV playlist: {wav_list_path.name}")
    return wav_list_path

# ============================================================================
# LOAD MASTER BLOCKS
# ============================================================================

print("\nLoading master blocks...")
master_block_1 = pd.read_csv(master_blocks_dir / "Master_Block_1.csv")
master_block_2 = pd.read_csv(master_blocks_dir / "Master_Block_2.csv")

print(f"✓ Master Block 1: {len(master_block_1)} trials")
print(f"✓ Master Block 2: {len(master_block_2)} trials")

# ============================================================================
# CREATE BLOCKS A, B, C from Master Block 1
# ============================================================================

print("\n" + "="*80)
print("CREATING BLOCKS A, B, C (from Master Block 1)")
print("="*80)

blocks_abc = {}
seeds_abc = [100, 200, 300]
names_abc = ['block_a', 'block_b', 'block_c']

for name, seed in zip(names_abc, seeds_abc):
    print(f"\n--- {name.upper()} ---")
    
    # Randomize
    block = constrained_randomization(
        master_block_1.drop('Trial_Number', axis=1), 
        seed=seed
    )
    
    # Add trial numbers
    block.insert(0, 'Trial_Number', range(1, len(block) + 1))
    
    # Verify audio files exist
    verify_audio_files(block, name)
    
    # Save CSV
    csv_path = output_dir / f"{name}.csv"
    block.to_csv(csv_path, index=False)
    print(f"  ✓ Saved CSV: {csv_path.name}")
    
    # Create WAV playlist
    create_wav_file_list(block, name, output_dir)
    
    blocks_abc[name] = block

# ============================================================================
# CREATE BLOCKS D, E, F from Master Block 2
# ============================================================================

print("\n" + "="*80)
print("CREATING BLOCKS D, E, F (from Master Block 2)")
print("="*80)

blocks_def = {}
seeds_def = [400, 500, 600]
names_def = ['block_d', 'block_e', 'block_f']

for name, seed in zip(names_def, seeds_def):
    print(f"\n--- {name.upper()} ---")
    
    # Randomize
    block = constrained_randomization(
        master_block_2.drop('Trial_Number', axis=1), 
        seed=seed
    )
    
    # Add trial numbers
    block.insert(0, 'Trial_Number', range(1, len(block) + 1))
    
    # Verify audio files exist
    verify_audio_files(block, name)
    
    # Save CSV
    csv_path = output_dir / f"{name}.csv"
    block.to_csv(csv_path, index=False)
    print(f"  ✓ Saved CSV: {csv_path.name}")
    
    # Create WAV playlist
    create_wav_file_list(block, name, output_dir)
    
    blocks_def[name] = block

# ============================================================================
# CREATE SUMMARY REPORT
# ============================================================================

print("\n" + "="*80)
print("GENERATING SUMMARY REPORT")
print("="*80)

all_blocks = {**blocks_abc, **blocks_def}
summary_data = []

for name, block in all_blocks.items():
    # Count trial types
    at_count = (block['Trial_Type'] == 'Audio-Tactile').sum()
    bl_count = (block['Trial_Type'] == 'Baseline').sum()
    ct_count = (block['Trial_Type'] == 'Catch').sum()
    
    # Count phases
    inhale_count = (block['Respiratory_Phase'] == 'Inhale').sum()
    exhale_count = (block['Respiratory_Phase'] == 'Exhale').sum()
    
    # Check max runs
    max_run_type = 0
    current_type = None
    current_run = 0
    for trial_type in block['Trial_Type']:
        if trial_type == current_type:
            current_run += 1
            max_run_type = max(max_run_type, current_run)
        else:
            current_type = trial_type
            current_run = 1
    
    # Catch spacing
    catch_indices = block[block['Trial_Type'] == 'Catch'].index.tolist()
    min_catch_spacing = None
    if len(catch_indices) > 1:
        spacings = [catch_indices[i+1] - catch_indices[i] for i in range(len(catch_indices)-1)]
        min_catch_spacing = min(spacings)
    
    summary_data.append({
        'Block': name.upper(),
        'Total_Trials': len(block),
        'Audio_Tactile': at_count,
        'Baseline': bl_count,
        'Catch': ct_count,
        'Inhale': inhale_count,
        'Exhale': exhale_count,
        'Max_Run_Length': max_run_type,
        'Min_Catch_Spacing': min_catch_spacing if min_catch_spacing else 'N/A'
    })

summary_df = pd.DataFrame(summary_data)

# Save summary
summary_path = output_dir / "experiment_blocks_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\n" + summary_df.to_string(index=False))
print(f"\n✓ Summary saved: {summary_path}")

# ============================================================================
# CREATE README
# ============================================================================

readme_path = output_dir / "README.txt"
with open(readme_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("EXPERIMENT BLOCKS - README\n")
    f.write("="*80 + "\n\n")
    
    f.write("CONTENTS:\n")
    f.write("---------\n")
    f.write("• 6 experiment blocks (A-F) as CSV files\n")
    f.write("• 6 WAV playlists for audio playback\n")
    f.write("• Summary report with block statistics\n\n")
    
    f.write("BLOCK STRUCTURE:\n")
    f.write("----------------\n")
    f.write("• Blocks A, B, C: Created from Master Block 1\n")
    f.write("• Blocks D, E, F: Created from Master Block 2\n")
    f.write("• Each block: 44 trials (22 Inhale, 22 Exhale)\n")
    f.write("• Perfect I-E-I-E alternation maintained\n\n")
    
    f.write("RANDOMIZATION CONSTRAINTS:\n")
    f.write("--------------------------\n")
    f.write("• Maximum run length: ≤3 same trial types\n")
    f.write("• Catch trial spacing: ≥8 trials apart\n")
    f.write("• Temporal balance: Equal distribution across quarters\n")
    f.write("• Respiratory phase: Perfect alternation (I-E-I-E...)\n\n")
    
    f.write("FILE FORMATS:\n")
    f.write("-------------\n")
    f.write("CSV files contain:\n")
    f.write("  - Trial_Number: Sequential trial number (1-44)\n")
    f.write("  - Trial_Type: Audio-Tactile, Baseline, or Catch\n")
    f.write("  - SOA_ms: Stimulus onset asynchrony in milliseconds\n")
    f.write("  - Noise_Type: Pink, Blue, White, Brown, or N/A\n")
    f.write("  - Respiratory_Phase: Inhale or Exhale\n\n")
    
    f.write("WAV playlists contain:\n")
    f.write("  - Ordered list of audio file paths\n")
    f.write("  - One file per trial\n")
    f.write("  - Ready for sequential playback\n\n")
    
    f.write("USAGE:\n")
    f.write("------\n")
    f.write("1. Assign participants to blocks in counterbalanced order\n")
    f.write("2. Load the corresponding CSV for trial parameters\n")
    f.write("3. Use WAV playlist for audio playback sequence\n")
    f.write("4. Each trial is exactly 8 seconds\n")
    f.write("5. Add inter-trial intervals as needed (e.g., 1000ms)\n\n")
    
    f.write("="*80 + "\n")

print(f"✓ README created: {readme_path}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("✓ EXPERIMENT BLOCKS GENERATED SUCCESSFULLY")
print("="*80)

print(f"\nOutput directory: {output_dir}")
print(f"\nFiles created:")
print(f"  • 6 CSV files (block_a.csv through block_f.csv)")
print(f"  • 6 WAV playlists (*_wav_playlist.txt)")
print(f"  • 1 Summary report (experiment_blocks_summary.csv)")
print(f"  • 1 README file (README.txt)")

print(f"\nTotal trials per block: 44")
print(f"Total unique blocks: 6")
print(f"Master blocks used: 2")

print("\n" + "="*80)

EXPERIMENT BLOCKS GENERATOR

Loading master blocks...
✓ Master Block 1: 44 trials
✓ Master Block 2: 44 trials

CREATING BLOCKS A, B, C (from Master Block 1)

--- BLOCK_A ---
  ✓ Valid sequence found in 72 attempts

  Verifying audio files for block_a...
  ✗ 4 audio files missing:
    Trial 20: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\6. Baseline\Exhale\Baseline_Exhale_SOA_2700ms.wav
    Trial 28: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\6. Baseline\Exhale\Baseline_Exhale_SOA_2700ms.wav
    Trial 35: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\6. Baseline\Inhale\Baseline_Inhale_SOA_2700ms.wav
    Trial 43: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\6. Baseline\Inhale\Baseline_Inhale_SOA_2700ms.wav
  ✓ Saved CSV: block_a.csv
  ✓ Created WAV playlist: block_a_wav_playlist.txt

--- BLOCK_B ---
  ✓ Valid sequence found in 164 attempts

  Verifying audio files for block_b...
  ✗ 4 audio files missing:


UnicodeEncodeError: 'charmap' codec can't encode character '\u2264' in position 22: character maps to <undefined>

================================================================================
8. 
================================================================================

In [39]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set base directory
base_dir = Path(r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli")
master_blocks_dir = base_dir / "8.Master_Blocks"
output_dir = master_blocks_dir  # Save in same folder

# ============================================================================
# HELPER FUNCTIONS FOR BEST-PRACTICE RANDOMIZATION
# ============================================================================

def check_max_runs(sequence, feature_col, max_run=3):
    """Check if any feature has more than max_run consecutive occurrences."""
    current_val = None
    current_run = 0
    
    for val in sequence[feature_col]:
        if val == current_val:
            current_run += 1
            if current_run > max_run:
                return False
        else:
            current_val = val
            current_run = 1
    return True

def check_catch_spacing(sequence, min_spacing=8):
    """Ensure catch trials are not too close together."""
    catch_indices = sequence[sequence['Trial_Type'] == 'Catch'].index.tolist()
    
    if len(catch_indices) < 2:
        return True
    
    for i in range(len(catch_indices) - 1):
        if catch_indices[i+1] - catch_indices[i] < min_spacing:
            return False
    return True

def check_temporal_balance(sequence, n_segments=4):
    """
    Check that trial types are balanced across temporal segments.
    Best practice: ensure each condition appears roughly equally in early vs. late portions.
    """
    segment_size = len(sequence) // n_segments
    
    # Check trial type balance across segments
    trial_types = ['Audio-Tactile', 'Baseline']  # Skip catch for balance
    
    for trial_type in trial_types:
        counts_per_segment = []
        for seg in range(n_segments):
            start_idx = seg * segment_size
            end_idx = (seg + 1) * segment_size if seg < n_segments - 1 else len(sequence)
            segment_data = sequence.iloc[start_idx:end_idx]
            count = (segment_data['Trial_Type'] == trial_type).sum()
            counts_per_segment.append(count)
        
        # Check if variance is acceptable (coefficient of variation < 0.5)
        mean_count = np.mean(counts_per_segment)
        if mean_count > 0:
            cv = np.std(counts_per_segment) / mean_count
            if cv > 0.5:
                return False
    
    return True

def constrained_randomization(trials_df, max_attempts=1000, seed=None):
    """
    Randomize trial order with constraints.
    
    Best practices implemented:
    1. Separate randomization by respiratory phase (inhale/exhale)
    2. Maximum run length constraints (≤3 same trial type)
    3. Catch trial spacing (≥8 trials apart)
    4. Temporal balance across segments
    
    IMPORTANT: Only randomizes ORDER, does not change trial composition.
    """
    if seed is not None:
        np.random.seed(seed)
    
    for attempt in range(max_attempts):
        # Separate by phase
        inhale_trials = trials_df[trials_df['Respiratory_Phase'] == 'Inhale'].copy()
        exhale_trials = trials_df[trials_df['Respiratory_Phase'] == 'Exhale'].copy()
        
        # Shuffle independently (only changes order, not content)
        inhale_trials = inhale_trials.sample(frac=1).reset_index(drop=True)
        exhale_trials = exhale_trials.sample(frac=1).reset_index(drop=True)
        
        # Interleave to maintain phase alternation
        randomized = []
        for i in range(max(len(inhale_trials), len(exhale_trials))):
            if i < len(inhale_trials):
                randomized.append(inhale_trials.iloc[i])
            if i < len(exhale_trials):
                randomized.append(exhale_trials.iloc[i])
        
        result = pd.DataFrame(randomized).reset_index(drop=True)
        
        # Apply constraints
        if not check_max_runs(result, 'Trial_Type', max_run=3):
            continue
        
        if not check_catch_spacing(result, min_spacing=8):
            continue
        
        if not check_temporal_balance(result, n_segments=4):
            continue
        
        # If all constraints satisfied, return
        print(f"  ✓ Valid sequence found in {attempt + 1} attempts")
        return result
    
    # If no valid sequence found, return best attempt with warning
    print(f"  ⚠ WARNING: Could not satisfy all constraints in {max_attempts} attempts")
    print(f"    Returning last attempt - manual inspection recommended")
    return result

def compute_diagnostics(sequence, block_name):
    """Compute and display diagnostic statistics for the randomization."""
    print(f"\n{'='*60}")
    print(f"DIAGNOSTICS FOR {block_name}")
    print(f"{'='*60}")
    
    # Trial type distribution
    print("\n1. Trial Type Distribution:")
    trial_counts = sequence['Trial_Type'].value_counts()
    for trial_type, count in trial_counts.items():
        print(f"   {trial_type}: {count}")
    
    # Run length analysis
    print("\n2. Maximum Run Lengths:")
    max_runs = {}
    for feature in ['Trial_Type', 'SOA_ms']:
        current_val = None
        current_run = 0
        max_run = 0
        
        for val in sequence[feature]:
            if val == current_val:
                current_run += 1
                max_run = max(max_run, current_run)
            else:
                current_val = val
                current_run = 1
        
        max_runs[feature] = max_run
        print(f"   {feature}: {max_run}")
    
    # Catch trial spacing
    print("\n3. Catch Trial Spacing:")
    catch_indices = sequence[sequence['Trial_Type'] == 'Catch'].index.tolist()
    spacings = []
    if len(catch_indices) > 1:
        spacings = [catch_indices[i+1] - catch_indices[i] for i in range(len(catch_indices)-1)]
        print(f"   Minimum spacing: {min(spacings)} trials")
        print(f"   Average spacing: {np.mean(spacings):.1f} trials")
    
    # Temporal balance (quarters)
    print("\n4. Temporal Balance (by quarters):")
    quarter_size = len(sequence) // 4
    for q in range(4):
        start_idx = q * quarter_size
        end_idx = (q + 1) * quarter_size if q < 3 else len(sequence)
        quarter = sequence.iloc[start_idx:end_idx]
        
        at_count = (quarter['Trial_Type'] == 'Audio-Tactile').sum()
        bl_count = (quarter['Trial_Type'] == 'Baseline').sum()
        ct_count = (quarter['Trial_Type'] == 'Catch').sum()
        
        print(f"   Quarter {q+1}: AT={at_count}, BL={bl_count}, Catch={ct_count}")
    
    return {
        'block_name': block_name,
        'total_trials': len(sequence),
        'max_run_trial_type': max_runs['Trial_Type'],
        'max_run_soa': max_runs['SOA_ms'],
        'min_catch_spacing': min(spacings) if len(spacings) > 0 else None
    }

# ============================================================================
# MAIN SCRIPT: LOAD MASTER BLOCKS AND CREATE RANDOMIZED VERSIONS
# ============================================================================

print("="*80)
print("COUNTERBALANCED BLOCK GENERATION")
print("="*80)

# Load master blocks
print("\nLoading master blocks...")
master_block_1 = pd.read_csv(master_blocks_dir / "Master_Block_1.csv")
master_block_2 = pd.read_csv(master_blocks_dir / "Master_Block_2.csv")

print(f"✓ Master Block 1: {len(master_block_1)} trials")
print(f"✓ Master Block 2: {len(master_block_2)} trials")

# ============================================================================
# CREATE BLOCKS A, B, C from Master Block 1 (3 different orders)
# ============================================================================

print("\n" + "="*80)
print("CREATING BLOCKS A, B, C (from Master Block 1)")
print("="*80)

# Block A - Seed 100
print("\n--- Block A ---")
block_a = constrained_randomization(master_block_1.drop('Trial_Number', axis=1), seed=100)
block_a.insert(0, 'Trial_Number', range(1, len(block_a) + 1))
diagnostics_a = compute_diagnostics(block_a, "Block A")

# Block B - Seed 200
print("\n--- Block B ---")
block_b = constrained_randomization(master_block_1.drop('Trial_Number', axis=1), seed=200)
block_b.insert(0, 'Trial_Number', range(1, len(block_b) + 1))
diagnostics_b = compute_diagnostics(block_b, "Block B")

# Block C - Seed 300
print("\n--- Block C ---")
block_c = constrained_randomization(master_block_1.drop('Trial_Number', axis=1), seed=300)
block_c.insert(0, 'Trial_Number', range(1, len(block_c) + 1))
diagnostics_c = compute_diagnostics(block_c, "Block C")

# ============================================================================
# CREATE BLOCKS D, E, F from Master Block 2 (3 different orders)
# ============================================================================

print("\n" + "="*80)
print("CREATING BLOCKS D, E, F (from Master Block 2)")
print("="*80)

# Block D - Seed 400
print("\n--- Block D ---")
block_d = constrained_randomization(master_block_2.drop('Trial_Number', axis=1), seed=400)
block_d.insert(0, 'Trial_Number', range(1, len(block_d) + 1))
diagnostics_d = compute_diagnostics(block_d, "Block D")

# Block E - Seed 500
print("\n--- Block E ---")
block_e = constrained_randomization(master_block_2.drop('Trial_Number', axis=1), seed=500)
block_e.insert(0, 'Trial_Number', range(1, len(block_e) + 1))
diagnostics_e = compute_diagnostics(block_e, "Block E")

# Block F - Seed 600
print("\n--- Block F ---")
block_f = constrained_randomization(master_block_2.drop('Trial_Number', axis=1), seed=600)
block_f.insert(0, 'Trial_Number', range(1, len(block_f) + 1))
diagnostics_f = compute_diagnostics(block_f, "Block F")

# ============================================================================
# SAVE ALL BLOCKS
# ============================================================================

print("\n" + "="*80)
print("SAVING BLOCKS")
print("="*80)

blocks = {
    'block_a.csv': block_a,
    'block_b.csv': block_b,
    'block_c.csv': block_c,
    'block_d.csv': block_d,
    'block_e.csv': block_e,
    'block_f.csv': block_f
}

for filename, block_df in blocks.items():
    filepath = output_dir / filename
    block_df.to_csv(filepath, index=False)
    print(f"✓ Saved {filepath}")

# ============================================================================
# SUMMARY DIAGNOSTICS ACROSS ALL BLOCKS
# ============================================================================

print("\n" + "="*80)
print("SUMMARY ACROSS ALL BLOCKS")
print("="*80)

all_diagnostics = [diagnostics_a, diagnostics_b, diagnostics_c, 
                   diagnostics_d, diagnostics_e, diagnostics_f]

summary_df = pd.DataFrame(all_diagnostics)
print("\n", summary_df.to_string(index=False))

# Save summary
summary_path = output_dir / "randomization_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"\n✓ Summary saved to {summary_path}")

# ============================================================================
# VERIFICATION: Ensure trial composition is preserved
# ============================================================================

print("\n" + "="*80)
print("VERIFICATION: Trial Composition Preserved")
print("="*80)

# Verify Blocks A, B, C have same composition as Master Block 1
print("\nMaster Block 1 Composition:")
mb1_composition = master_block_1.groupby(['Trial_Type', 'SOA_ms', 'Noise_Type', 'Respiratory_Phase']).size().reset_index(name='count')
print(f"  Unique combinations: {len(mb1_composition)}")

for block_name, block_df in [('A', block_a), ('B', block_b), ('C', block_c)]:
    block_composition = block_df.groupby(['Trial_Type', 'SOA_ms', 'Noise_Type', 'Respiratory_Phase']).size().reset_index(name='count')
    
    # Sort both for comparison
    mb1_sorted = mb1_composition.sort_values(['Trial_Type', 'SOA_ms', 'Noise_Type', 'Respiratory_Phase']).reset_index(drop=True)
    block_sorted = block_composition.sort_values(['Trial_Type', 'SOA_ms', 'Noise_Type', 'Respiratory_Phase']).reset_index(drop=True)
    
    if mb1_sorted.equals(block_sorted):
        print(f"✓ Block {block_name}: Composition matches Master Block 1")
    else:
        print(f"✗ Block {block_name}: ERROR - Composition does NOT match!")

# Verify Blocks D, E, F have same composition as Master Block 2
print("\nMaster Block 2 Composition:")
mb2_composition = master_block_2.groupby(['Trial_Type', 'SOA_ms', 'Noise_Type', 'Respiratory_Phase']).size().reset_index(name='count')
print(f"  Unique combinations: {len(mb2_composition)}")

for block_name, block_df in [('D', block_d), ('E', block_e), ('F', block_f)]:
    block_composition = block_df.groupby(['Trial_Type', 'SOA_ms', 'Noise_Type', 'Respiratory_Phase']).size().reset_index(name='count')
    
    # Sort both for comparison
    mb2_sorted = mb2_composition.sort_values(['Trial_Type', 'SOA_ms', 'Noise_Type', 'Respiratory_Phase']).reset_index(drop=True)
    block_sorted = block_composition.sort_values(['Trial_Type', 'SOA_ms', 'Noise_Type', 'Respiratory_Phase']).reset_index(drop=True)
    
    if mb2_sorted.equals(block_sorted):
        print(f"✓ Block {block_name}: Composition matches Master Block 2")
    else:
        print(f"✗ Block {block_name}: ERROR - Composition does NOT match!")

print("\n" + "="*80)
print("✓ COUNTERBALANCED BLOCK GENERATION COMPLETE")
print("="*80)
print(f"\nAll files saved to: {output_dir}")
print("\nWhat was done:")
print("  • Master Block 1 → 3 uniquely ordered versions (A, B, C)")
print("  • Master Block 2 → 3 uniquely ordered versions (D, E, F)")
print("  • Trial composition preserved exactly")
print("  • Only ORDER randomized with constraints")
print("\nBest practices implemented:")
print("  ✓ Maximum run-length control (≤3 same trial types)")
print("  ✓ Catch trial spacing enforcement (≥8 trials)")
print("  ✓ Temporal balance across quarters")
print("  ✓ Independent respiratory phase randomization")
print("  ✓ Reproducible seeding per block")

COUNTERBALANCED BLOCK GENERATION

Loading master blocks...
✓ Master Block 1: 44 trials
✓ Master Block 2: 44 trials

CREATING BLOCKS A, B, C (from Master Block 1)

--- Block A ---
  ✓ Valid sequence found in 375 attempts

DIAGNOSTICS FOR Block A

1. Trial Type Distribution:
   Baseline: 20
   Audio-Tactile: 20
   Catch: 4

2. Maximum Run Lengths:
   Trial_Type: 3
   SOA_ms: 2

3. Catch Trial Spacing:
   Minimum spacing: 11 trials
   Average spacing: 14.3 trials

4. Temporal Balance (by quarters):
   Quarter 1: AT=4, BL=6, Catch=1
   Quarter 2: AT=5, BL=5, Catch=1
   Quarter 3: AT=5, BL=5, Catch=1
   Quarter 4: AT=6, BL=4, Catch=1

--- Block B ---
  ✓ Valid sequence found in 177 attempts

DIAGNOSTICS FOR Block B

1. Trial Type Distribution:
   Audio-Tactile: 20
   Baseline: 20
   Catch: 4

2. Maximum Run Lengths:
   Trial_Type: 3
   SOA_ms: 3

3. Catch Trial Spacing:
   Minimum spacing: 10 trials
   Average spacing: 13.0 trials

4. Temporal Balance (by quarters):
   Quarter 1: AT=6, BL=4

================================================================================
9. EXPERIMENT BLOCKS GENERATOR 
================================================================================

In [1]:
import pandas as pd
from pathlib import Path
import soundfile as sf

# Set base directory
base_dir = Path(r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli")
master_blocks_dir = base_dir / "8.Master_Blocks"
output_dir = base_dir / "9.Experiment_Blocks"
output_dir.mkdir(exist_ok=True)

# Audio file directories
audio_dir_inhale = base_dir / "5. Breathingphase+LoomingXTactile" / "Inhale"
audio_dir_exhale = base_dir / "5. Breathingphase+LoomingXTactile" / "Exhale"
baseline_dir_inhale = base_dir / "6. Baseline" / "Inhale"
baseline_dir_exhale = base_dir / "6. Baseline" / "Exhale"
catch_dir_inhale = base_dir / "7. Catch" / "Inhale"
catch_dir_exhale = base_dir / "7. Catch" / "Exhale"

SAMPLE_RATE = 44100

print("="*80)
print("EXPERIMENT BLOCKS GENERATOR WITH CONCATENATED WAV FILES (NO ITI)")
print("="*80)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def check_max_runs(sequence, feature_col, max_run=3):
    """Check if any feature has more than max_run consecutive occurrences."""
    current_val = None
    current_run = 0
    
    for val in sequence[feature_col]:
        if val == current_val:
            current_run += 1
            if current_run > max_run:
                return False
        else:
            current_val = val
            current_run = 1
    return True

def check_catch_spacing(sequence, min_spacing=8):
    """Ensure catch trials are not too close together."""
    catch_indices = sequence[sequence['Trial_Type'] == 'Catch'].index.tolist()
    
    if len(catch_indices) < 2:
        return True
    
    for i in range(len(catch_indices) - 1):
        if catch_indices[i+1] - catch_indices[i] < min_spacing:
            return False
    return True

def check_temporal_balance(sequence, n_segments=4):
    """Check that trial types are balanced across temporal segments."""
    segment_size = len(sequence) // n_segments
    trial_types = ['Audio-Tactile', 'Baseline']
    
    for trial_type in trial_types:
        counts_per_segment = []
        for seg in range(n_segments):
            start_idx = seg * segment_size
            end_idx = (seg + 1) * segment_size if seg < n_segments - 1 else len(sequence)
            segment_data = sequence.iloc[start_idx:end_idx]
            count = (segment_data['Trial_Type'] == trial_type).sum()
            counts_per_segment.append(count)
        
        if len(counts_per_segment) > 0:
            mean_count = sum(counts_per_segment) / len(counts_per_segment)
            if mean_count > 0:
                # Calculate coefficient of variation manually
                variance = sum((x - mean_count) ** 2 for x in counts_per_segment) / len(counts_per_segment)
                std = variance ** 0.5
                cv = std / mean_count
                if cv > 0.5:
                    return False
    
    return True

def constrained_randomization(trials_df, max_attempts=1000, seed=None):
    """
    Randomize trial order with constraints while maintaining I-E-I-E alternation.
    """
    import random
    if seed is not None:
        random.seed(seed)
    
    for attempt in range(max_attempts):
        # Separate by phase
        inhale_trials = trials_df[trials_df['Respiratory_Phase'] == 'Inhale'].copy()
        exhale_trials = trials_df[trials_df['Respiratory_Phase'] == 'Exhale'].copy()
        
        # Shuffle independently
        inhale_trials = inhale_trials.sample(frac=1, random_state=seed+attempt if seed else None).reset_index(drop=True)
        exhale_trials = exhale_trials.sample(frac=1, random_state=seed+attempt+1000 if seed else None).reset_index(drop=True)
        
        # Interleave to maintain phase alternation
        randomized = []
        for i in range(max(len(inhale_trials), len(exhale_trials))):
            if i < len(inhale_trials):
                randomized.append(inhale_trials.iloc[i])
            if i < len(exhale_trials):
                randomized.append(exhale_trials.iloc[i])
        
        result = pd.DataFrame(randomized).reset_index(drop=True)
        
        # Apply constraints
        if not check_max_runs(result, 'Trial_Type', max_run=3):
            continue
        
        if not check_catch_spacing(result, min_spacing=8):
            continue
        
        if not check_temporal_balance(result, n_segments=4):
            continue
        
        print(f"  ✓ Valid sequence found in {attempt + 1} attempts")
        return result
    
    print(f"  ⚠ WARNING: Could not satisfy all constraints in {max_attempts} attempts")
    return result

def get_audio_filename(row):
    """
    Generate the expected audio filename based on trial parameters.
    """
    trial_type = row['Trial_Type']
    soa = row['SOA_ms']
    noise = row['Noise_Type']
    phase = row['Respiratory_Phase']
    
    if trial_type == 'Audio-Tactile':
        noise_map = {'Pink': '1-pink', 'Blue': '2-blue', 'White': '3-white', 'Brown': '4-brown'}
        noise_code = noise_map[noise]
        filename = f"Stereo_LEFT_tactile_stimulus_{soa}ms__RIGHT_Loom-{noise_code}.wav"
        
    elif trial_type == 'Baseline':
        filename = f"Baseline_{phase}_SOA_{soa}ms.wav"
        
    elif trial_type == 'Catch':
        noise_lower = noise.lower()
        filename = f"Catch_{phase}_SOA_{soa}ms_{noise_lower}.wav"
    
    return filename

def get_audio_filepath(row):
    """
    Get the full path to the audio file.
    """
    filename = get_audio_filename(row)
    trial_type = row['Trial_Type']
    phase = row['Respiratory_Phase']
    
    if trial_type == 'Audio-Tactile':
        if phase == 'Inhale':
            return audio_dir_inhale / filename
        else:
            return audio_dir_exhale / filename
            
    elif trial_type == 'Baseline':
        if phase == 'Inhale':
            return baseline_dir_inhale / filename
        else:
            return baseline_dir_exhale / filename
            
    elif trial_type == 'Catch':
        if phase == 'Inhale':
            return catch_dir_inhale / filename
        else:
            return catch_dir_exhale / filename
    
    return None

def concatenate_wav_files(df, block_name, output_dir, sample_rate=44100):
    """
    Concatenate all WAV files for a block WITHOUT inter-trial intervals.
    """
    print(f"\n  Concatenating WAV files for {block_name} (NO ITI)...")
    
    import numpy as np
    concatenated_audio = []
    missing_files = []
    
    for idx, row in df.iterrows():
        filepath = get_audio_filepath(row)
        
        if filepath is None or not filepath.exists():
            missing_files.append((idx + 1, filepath))
            print(f"    ✗ Trial {idx + 1}: File not found - {filepath}")
            continue
        
        # Load audio file
        audio, sr = sf.read(filepath)
        
        # Ensure stereo
        if len(audio.shape) == 1:
            audio = np.column_stack([audio, audio])
        
        # Resample if needed
        if sr != sample_rate:
            print(f"    ⚠ Trial {idx + 1}: Resampling from {sr} to {sample_rate} Hz")
            from scipy import signal
            num_samples = int(len(audio) * sample_rate / sr)
            audio = np.array([
                signal.resample(audio[:, ch], num_samples)
                for ch in range(audio.shape[1])
            ]).T
        
        # Add to concatenated audio (NO ITI!)
        concatenated_audio.append(audio)
    
    if len(missing_files) > 0:
        print(f"    ⚠ WARNING: {len(missing_files)} files missing, block incomplete")
        return None, missing_files
    
    # Concatenate all audio
    final_audio = np.vstack(concatenated_audio)
    
    # Save concatenated WAV file
    output_path = output_dir / f"{block_name}_concatenated.wav"
    sf.write(output_path, final_audio, sample_rate)
    
    duration_seconds = len(final_audio) / sample_rate
    duration_minutes = duration_seconds / 60
    
    print(f"  ✓ Concatenated WAV saved: {output_path.name}")
    print(f"    Duration: {duration_minutes:.2f} minutes ({duration_seconds:.1f} seconds)")
    print(f"    Trials: {len(df)}, NO ITI (back-to-back)")
    
    return output_path, []

# ============================================================================
# LOAD MASTER BLOCKS
# ============================================================================

print("\nLoading master blocks...")
master_block_1 = pd.read_csv(master_blocks_dir / "Master_Block_1.csv")
master_block_2 = pd.read_csv(master_blocks_dir / "Master_Block_2.csv")

print(f"✓ Master Block 1: {len(master_block_1)} trials")
print(f"✓ Master Block 2: {len(master_block_2)} trials")

# ============================================================================
# CREATE BLOCKS A, B, C from Master Block 1
# ============================================================================

print("\n" + "="*80)
print("CREATING BLOCKS A, B, C (from Master Block 1)")
print("="*80)

blocks_abc = {}
seeds_abc = [100, 200, 300]
names_abc = ['block_a', 'block_b', 'block_c']

for name, seed in zip(names_abc, seeds_abc):
    print(f"\n--- {name.upper()} ---")
    
    # Randomize
    block = constrained_randomization(
        master_block_1.drop('Trial_Number', axis=1), 
        seed=seed
    )
    
    # Add trial numbers
    block.insert(0, 'Trial_Number', range(1, len(block) + 1))
    
    # Save CSV
    csv_path = output_dir / f"{name}.csv"
    block.to_csv(csv_path, index=False)
    print(f"  ✓ Saved CSV: {csv_path.name}")
    
    # Concatenate WAV files (NO ITI)
    wav_path, missing = concatenate_wav_files(block, name, output_dir, SAMPLE_RATE)
    
    blocks_abc[name] = block

# ============================================================================
# CREATE BLOCKS D, E, F from Master Block 2
# ============================================================================

print("\n" + "="*80)
print("CREATING BLOCKS D, E, F (from Master Block 2)")
print("="*80)

blocks_def = {}
seeds_def = [400, 500, 600]
names_def = ['block_d', 'block_e', 'block_f']

for name, seed in zip(names_def, seeds_def):
    print(f"\n--- {name.upper()} ---")
    
    # Randomize
    block = constrained_randomization(
        master_block_2.drop('Trial_Number', axis=1), 
        seed=seed
    )
    
    # Add trial numbers
    block.insert(0, 'Trial_Number', range(1, len(block) + 1))
    
    # Save CSV
    csv_path = output_dir / f"{name}.csv"
    block.to_csv(csv_path, index=False)
    print(f"  ✓ Saved CSV: {csv_path.name}")
    
    # Concatenate WAV files (NO ITI)
    wav_path, missing = concatenate_wav_files(block, name, output_dir, SAMPLE_RATE)
    
    blocks_def[name] = block

# ============================================================================
# CREATE SUMMARY REPORT
# ============================================================================

print("\n" + "="*80)
print("GENERATING SUMMARY REPORT")
print("="*80)

all_blocks = {**blocks_abc, **blocks_def}
summary_data = []

for name, block in all_blocks.items():
    at_count = (block['Trial_Type'] == 'Audio-Tactile').sum()
    bl_count = (block['Trial_Type'] == 'Baseline').sum()
    ct_count = (block['Trial_Type'] == 'Catch').sum()
    
    inhale_count = (block['Respiratory_Phase'] == 'Inhale').sum()
    exhale_count = (block['Respiratory_Phase'] == 'Exhale').sum()
    
    max_run_type = 0
    current_type = None
    current_run = 0
    for trial_type in block['Trial_Type']:
        if trial_type == current_type:
            current_run += 1
            max_run_type = max(max_run_type, current_run)
        else:
            current_type = trial_type
            current_run = 1
    
    catch_indices = block[block['Trial_Type'] == 'Catch'].index.tolist()
    min_catch_spacing = None
    if len(catch_indices) > 1:
        spacings = [catch_indices[i+1] - catch_indices[i] for i in range(len(catch_indices)-1)]
        min_catch_spacing = min(spacings)
    
    summary_data.append({
        'Block': name.upper(),
        'Total_Trials': len(block),
        'Audio_Tactile': at_count,
        'Baseline': bl_count,
        'Catch': ct_count,
        'Inhale': inhale_count,
        'Exhale': exhale_count,
        'Max_Run_Length': max_run_type,
        'Min_Catch_Spacing': min_catch_spacing if min_catch_spacing else 'N/A'
    })

summary_df = pd.DataFrame(summary_data)

summary_path = output_dir / "experiment_blocks_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\n" + summary_df.to_string(index=False))
print(f"\n✓ Summary saved: {summary_path}")

# ============================================================================
# CREATE README
# ============================================================================

readme_path = output_dir / "README.txt"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("EXPERIMENT BLOCKS - README\n")
    f.write("="*80 + "\n\n")
    
    f.write("CONTENTS:\n")
    f.write("---------\n")
    f.write("- 6 experiment blocks (A-F) as CSV files\n")
    f.write("- 6 concatenated WAV files (one per block)\n")
    f.write("- Summary report with block statistics\n\n")
    
    f.write("BLOCK STRUCTURE:\n")
    f.write("----------------\n")
    f.write("- Blocks A, B, C: Created from Master Block 1\n")
    f.write("- Blocks D, E, F: Created from Master Block 2\n")
    f.write("- Each block: 44 trials (22 Inhale, 22 Exhale)\n")
    f.write("- Perfect I-E-I-E alternation maintained\n")
    f.write("- NO inter-trial intervals (trials play back-to-back)\n\n")
    
    f.write("WAV FILES:\n")
    f.write("----------\n")
    f.write("Each block has a concatenated WAV file:\n")
    f.write("  - block_a_concatenated.wav\n")
    f.write("  - block_b_concatenated.wav\n")
    f.write("  - etc.\n\n")
    f.write("Structure: [Trial 1 (8s)][Trial 2 (8s)][Trial 3 (8s)]... (NO gaps)\n")
    f.write(f"Total duration per block: {44 * 8} seconds ({44 * 8 / 60:.1f} minutes)\n\n")
    
    f.write("RANDOMIZATION CONSTRAINTS:\n")
    f.write("--------------------------\n")
    f.write("- Maximum run length: <=3 same trial types\n")
    f.write("- Catch trial spacing: >=8 trials apart\n")
    f.write("- Temporal balance: Equal distribution across quarters\n")
    f.write("- Respiratory phase: Perfect alternation (I-E-I-E...)\n\n")
    
    f.write("="*80 + "\n")

print(f"✓ README created: {readme_path}")

print("\n" + "="*80)
print("✓ EXPERIMENT BLOCKS GENERATED SUCCESSFULLY")
print("="*80)

print(f"\nOutput directory: {output_dir}")
print(f"\nFiles created:")
print(f"  - 6 CSV files (block_a.csv through block_f.csv)")
print(f"  - 6 Concatenated WAV files (block_*_concatenated.wav)")
print(f"  - 1 Summary report (experiment_blocks_summary.csv)")
print(f"  - 1 README file (README.txt)")

print(f"\nNOTE: NO inter-trial intervals - all trials play back-to-back")
print(f"Block duration: {44 * 8} seconds ({44 * 8 / 60:.1f} minutes) each")

print("\n" + "="*80)

EXPERIMENT BLOCKS GENERATOR WITH CONCATENATED WAV FILES (NO ITI)

Loading master blocks...
✓ Master Block 1: 44 trials
✓ Master Block 2: 44 trials

CREATING BLOCKS A, B, C (from Master Block 1)

--- BLOCK_A ---
  ✓ Valid sequence found in 85 attempts
  ✓ Saved CSV: block_a.csv

  Concatenating WAV files for block_a (NO ITI)...
  ✓ Concatenated WAV saved: block_a_concatenated.wav
    Duration: 5.87 minutes (352.0 seconds)
    Trials: 44, NO ITI (back-to-back)

--- BLOCK_B ---
  ✓ Valid sequence found in 32 attempts
  ✓ Saved CSV: block_b.csv

  Concatenating WAV files for block_b (NO ITI)...
  ✓ Concatenated WAV saved: block_b_concatenated.wav
    Duration: 5.87 minutes (352.0 seconds)
    Trials: 44, NO ITI (back-to-back)

--- BLOCK_C ---
  ✓ Valid sequence found in 16 attempts
  ✓ Saved CSV: block_c.csv

  Concatenating WAV files for block_c (NO ITI)...
  ✓ Concatenated WAV saved: block_c_concatenated.wav
    Duration: 5.87 minutes (352.0 seconds)
    Trials: 44, NO ITI (back-to-back)

================================================================================
10. PARTICIPANT SEQUENCE GENERATOR
================================================================================

In [5]:
import pandas as pd
from pathlib import Path
import shutil
from itertools import permutations
import random

# Set base directory
base_dir = Path(r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli")
blocks_dir = base_dir / "9.Experiment_Blocks"
output_dir = base_dir / "10.Participant_Sequences"
output_dir.mkdir(exist_ok=True)

print("="*80)
print("PARTICIPANT-SPECIFIC SEQUENCE GENERATOR")
print("="*80)

# ============================================================================
# VERIFY BLOCK FILES EXIST
# ============================================================================

print("\nVerifying experiment blocks...")

block_names = ['block_a', 'block_b', 'block_c', 'block_d', 'block_e', 'block_f']

# Check that all blocks exist
for block_name in block_names:
    csv_path = blocks_dir / f"{block_name}.csv"
    wav_path = blocks_dir / f"{block_name}_concatenated.wav"
    
    if not csv_path.exists():
        print(f"✗ ERROR: {csv_path} not found")
        exit(1)
    if not wav_path.exists():
        print(f"✗ ERROR: {wav_path} not found")
        exit(1)
    
    print(f"✓ Found {block_name}.csv and {block_name}_concatenated.wav")

# ============================================================================
# GENERATE 50 UNIQUE BLOCK ORDERS
# ============================================================================

print("\n" + "="*80)
print("GENERATING 50 UNIQUE BLOCK ORDERS")
print("="*80)

# All possible permutations of 6 blocks = 6! = 720 possible orders
all_permutations = list(permutations(block_names))
print(f"\nTotal possible orders: {len(all_permutations)}")

# Randomly select 50 unique orders
random.seed(42)  # For reproducibility
selected_orders = random.sample(all_permutations, 50)

print(f"Selected 50 unique orders for participants")

# Save the order assignments
orders_data = []
for i in range(50):
    participant_id = f'P{i+1:02d}'
    block_order = '-'.join([b.split('_')[1].upper() for b in selected_orders[i]])
    orders_data.append({
        'Participant_ID': participant_id,
        'Block_Order': block_order
    })

orders_df = pd.DataFrame(orders_data)
orders_path = output_dir / "participant_block_orders.csv"
orders_df.to_csv(orders_path, index=False)
print(f"✓ Saved block orders: {orders_path}")

# Display first 10 orders as example
print("\nExample block orders:")
for idx in range(min(10, len(orders_df))):
    print(f"  {orders_df.iloc[idx]['Participant_ID']}: {orders_df.iloc[idx]['Block_Order']}")

# ============================================================================
# CREATE PARTICIPANT FOLDERS AND FILES
# ============================================================================

print("\n" + "="*80)
print("CREATING PARTICIPANT FILES")
print("="*80)

for participant_idx, block_order in enumerate(selected_orders, start=1):
    participant_id = f"P{participant_idx:02d}"
    print(f"\n[{participant_idx}/50] Processing {participant_id}...")
    
    # Create participant folder
    participant_dir = output_dir / participant_id
    participant_dir.mkdir(exist_ok=True)
    
    # Extract block letters (a, b, c, etc.)
    block_letters = [b.split('_')[1] for b in block_order]
    order_string = '-'.join([l.upper() for l in block_letters])
    print(f"  Block order: {order_string}")
    
    # ========================================================================
    # Copy and rename block files
    # ========================================================================
    
    combined_trials = []
    
    for block_position, block_name in enumerate(block_order, start=1):
        block_letter = block_name.split('_')[1]  # 'a', 'b', 'c', etc.
        
        # New filename format: "1a", "2f", "3b", etc.
        new_prefix = f"{block_position}{block_letter}"
        
        # ====================================================================
        # COPY CSV (with added Block_Number and Block_Letter columns)
        # ====================================================================
        source_csv = blocks_dir / f"{block_name}.csv"
        dest_csv = participant_dir / f"{new_prefix}.csv"
        
        # Load CSV, add block info columns
        block_df = pd.read_csv(source_csv)
        block_df.insert(1, 'Block_Letter', block_letter.upper())
        block_df.insert(2, 'Block_Number', block_position)
        
        # Save with new name
        block_df.to_csv(dest_csv, index=False)
        print(f"  ✓ Created {new_prefix}.csv")
        
        # Collect for combined CSV
        combined_trials.append(block_df)
        
        # ====================================================================
        # COPY WAV (just rename, no modification needed)
        # ====================================================================
        source_wav = blocks_dir / f"{block_name}_concatenated.wav"
        dest_wav = participant_dir / f"{new_prefix}_concatenated.wav"
        
        shutil.copy2(source_wav, dest_wav)
        print(f"  ✓ Copied {new_prefix}_concatenated.wav")
    
    # ========================================================================
    # Create combined CSV with all trials
    # ========================================================================
    
    # Combine all blocks
    participant_csv = pd.concat(combined_trials, ignore_index=True)
    
    # Renumber trials globally
    participant_csv['Trial_Number'] = range(1, len(participant_csv) + 1)
    
    # Reorder columns for clarity
    cols = ['Trial_Number', 'Block_Number', 'Block_Letter', 'Trial_Type', 
            'SOA_ms', 'Noise_Type', 'Respiratory_Phase']
    participant_csv = participant_csv[cols]
    
    # Save combined CSV
    combined_csv_path = participant_dir / f"{participant_id}_full_sequence.csv"
    participant_csv.to_csv(combined_csv_path, index=False)
    
    print(f"  ✓ Created {participant_id}_full_sequence.csv ({len(participant_csv)} trials)")
    
    # ========================================================================
    # Create summary file
    # ========================================================================
    
    summary_path = participant_dir / f"{participant_id}_summary.txt"
    with open(summary_path, 'w', encoding='utf-8') as f:
        f.write(f"PARTICIPANT {participant_id} - EXPERIMENT SUMMARY\n")
        f.write("="*60 + "\n\n")
        
        f.write(f"Block Order: {order_string}\n\n")
        
        f.write("Block Sequence:\n")
        for block_position, block_name in enumerate(block_order, start=1):
            block_letter = block_name.split('_')[1].upper()
            trials_count = (participant_csv['Block_Number'] == block_position).sum()
            f.write(f"  {block_position}{block_letter} - {trials_count} trials\n")
        
        f.write(f"\nTotal Trials: {len(participant_csv)}\n")
        
        # Trial type breakdown
        f.write(f"\nTrial Type Distribution:\n")
        for trial_type in ['Audio-Tactile', 'Baseline', 'Catch']:
            count = (participant_csv['Trial_Type'] == trial_type).sum()
            f.write(f"  {trial_type}: {count}\n")
        
        # Respiratory phase breakdown
        f.write(f"\nRespiratory Phase Distribution:\n")
        inhale = (participant_csv['Respiratory_Phase'] == 'Inhale').sum()
        exhale = (participant_csv['Respiratory_Phase'] == 'Exhale').sum()
        f.write(f"  Inhale: {inhale}\n")
        f.write(f"  Exhale: {exhale}\n")
        
        f.write(f"\nEstimated Duration:\n")
        f.write(f"  Trial duration: 8 seconds each\n")
        f.write(f"  No inter-trial intervals\n")
        f.write(f"  Per block: {44 * 8} seconds ({44 * 8 / 60:.1f} minutes)\n")
        f.write(f"  Total time: {len(participant_csv) * 8} seconds ({(len(participant_csv) * 8) / 60:.1f} minutes)\n")
    
    print(f"  ✓ Created {participant_id}_summary.txt")

# ============================================================================
# CREATE MASTER README
# ============================================================================

readme_path = output_dir / "README.txt"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("PARTICIPANT-SPECIFIC SEQUENCES - README\n")
    f.write("="*80 + "\n\n")
    
    f.write("DIRECTORY STRUCTURE:\n")
    f.write("--------------------\n")
    f.write("10.Participant_Sequences/\n")
    f.write("├── participant_block_orders.csv (Master list of all assignments)\n")
    f.write("├── README.txt (This file)\n")
    f.write("├── P01/\n")
    f.write("│   ├── 1a.csv\n")
    f.write("│   ├── 1a_concatenated.wav\n")
    f.write("│   ├── 2f.csv\n")
    f.write("│   ├── 2f_concatenated.wav\n")
    f.write("│   ├── ... (blocks 3-6)\n")
    f.write("│   ├── P01_full_sequence.csv (All 264 trials combined)\n")
    f.write("│   └── P01_summary.txt\n")
    f.write("├── P02/\n")
    f.write("│   └── ...\n")
    f.write("└── ... (P03 through P50)\n\n")
    
    f.write("FILE NAMING:\n")
    f.write("------------\n")
    f.write("Block files are named with: [position][letter]\n")
    f.write("  Examples: 1a, 2f, 3b, 4c, 5d, 6e\n")
    f.write("  - First number (1-6): Order in which block is presented\n")
    f.write("  - Letter (a-f): Original block identity\n\n")
    
    f.write("CONTENTS PER PARTICIPANT FOLDER:\n")
    f.write("--------------------------------\n")
    f.write("1. Six block CSV files (e.g., 1a.csv, 2f.csv, etc.)\n")
    f.write("   - Contains trial-by-trial parameters\n")
    f.write("   - Includes Block_Number and Block_Letter columns\n")
    f.write("   - 44 trials per block\n\n")
    
    f.write("2. Six concatenated WAV files (e.g., 1a_concatenated.wav)\n")
    f.write("   - Audio for entire block\n")
    f.write("   - NO inter-trial intervals\n")
    f.write("   - Trials play back-to-back\n")
    f.write("   - Duration: 352 seconds (5.9 minutes) per block\n\n")
    
    f.write("3. Full sequence CSV (e.g., P01_full_sequence.csv)\n")
    f.write("   - All 264 trials combined\n")
    f.write("   - Global trial numbering (1-264)\n")
    f.write("   - Includes block number and letter for each trial\n\n")
    
    f.write("4. Summary text file (e.g., P01_summary.txt)\n")
    f.write("   - Block order for this participant\n")
    f.write("   - Trial counts and distributions\n")
    f.write("   - Estimated duration\n\n")
    
    f.write("COUNTERBALANCING:\n")
    f.write("-----------------\n")
    f.write("- 50 unique block orders\n")
    f.write("- Randomly sampled from 720 possible permutations (6!)\n")
    f.write("- Ensures balanced exposure across participants\n")
    f.write("- See participant_block_orders.csv for complete list\n\n")
    
    f.write("CSV COLUMNS (Full Sequence):\n")
    f.write("----------------------------\n")
    f.write("- Trial_Number: Global trial number (1-264)\n")
    f.write("- Block_Number: Presentation order (1-6)\n")
    f.write("- Block_Letter: Original block identity (A-F)\n")
    f.write("- Trial_Type: Audio-Tactile, Baseline, or Catch\n")
    f.write("- SOA_ms: Stimulus onset asynchrony in milliseconds\n")
    f.write("- Noise_Type: Pink, Blue, White, Brown, or N/A\n")
    f.write("- Respiratory_Phase: Inhale or Exhale\n\n")
    
    f.write("USAGE:\n")
    f.write("------\n")
    f.write("1. Assign participant to next available ID (P01-P50)\n")
    f.write("2. Open corresponding participant folder\n")
    f.write("3. Run blocks in numerical order (1→2→3→4→5→6)\n")
    f.write("4. Use CSV files for trial parameters\n")
    f.write("5. Play corresponding WAV files for audio\n")
    f.write("6. Reference full_sequence.csv for complete trial list\n\n")
    
    f.write("EXPERIMENT PARAMETERS:\n")
    f.write("----------------------\n")
    f.write("- Total trials per participant: 264 (44 × 6 blocks)\n")
    f.write("- Trial duration: 8 seconds\n")
    f.write("- Inter-trial interval: NONE (back-to-back trials)\n")
    f.write("- Block duration: 352 seconds (5.9 minutes)\n")
    f.write("- Total experiment duration: 2112 seconds (35.2 minutes)\n")
    f.write("- Blocks can be run with breaks in between\n\n")
    
    f.write("="*80 + "\n")

print(f"\n✓ Master README created: {readme_path}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("✓ PARTICIPANT SEQUENCES GENERATED SUCCESSFULLY")
print("="*80)

print(f"\nOutput directory: {output_dir}")
print(f"\nGenerated:")
print(f"  - 50 participant folders (P01 through P50)")
print(f"  - Each folder contains:")
print(f"    * 6 renumbered block CSV files (e.g., 1a.csv, 2f.csv)")
print(f"    * 6 renumbered block WAV files (copied from folder 9)")
print(f"    * 1 full sequence CSV (all 264 trials)")
print(f"    * 1 summary text file")
print(f"  - 1 master block order assignment file")
print(f"  - 1 master README")

print(f"\nFile naming examples:")
print(f"  - If order is A-F-B-C-D-E:")
print(f"    * 1a.csv, 1a_concatenated.wav")
print(f"    * 2f.csv, 2f_concatenated.wav")
print(f"    * 3b.csv, 3b_concatenated.wav")
print(f"    * etc.")

print(f"\nExperiment structure:")
print(f"  - Participants: 50")
print(f"  - Trials per participant: 264")
print(f"  - Blocks per participant: 6")
print(f"  - Unique block orders: 50")
print(f"  - Duration: ~35.2 minutes per participant")

print("\n" + "="*80)

PARTICIPANT-SPECIFIC SEQUENCE GENERATOR

Verifying experiment blocks...
✓ Found block_a.csv and block_a_concatenated.wav
✓ Found block_b.csv and block_b_concatenated.wav
✓ Found block_c.csv and block_c_concatenated.wav
✓ Found block_d.csv and block_d_concatenated.wav
✓ Found block_e.csv and block_e_concatenated.wav
✓ Found block_f.csv and block_f_concatenated.wav

GENERATING 50 UNIQUE BLOCK ORDERS

Total possible orders: 720
Selected 50 unique orders for participants
✓ Saved block orders: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\10.Participant_Sequences\participant_block_orders.csv

Example block orders:
  P01: F-C-B-A-D-E
  P02: A-F-E-B-C-D
  P03: A-C-B-D-F-E
  P04: C-B-E-F-D-A
  P05: C-A-D-F-B-E
  P06: B-F-D-A-C-E
  P07: B-A-F-E-C-D
  P08: A-F-C-D-B-E
  P09: F-D-E-B-A-C
  P10: E-D-B-A-C-F

CREATING PARTICIPANT FILES

[1/50] Processing P01...
  Block order: F-C-B-A-D-E
  ✓ Created 1f.csv
  ✓ Copied 1f_concatenated.wav
  ✓ Created 2c.csv
  ✓ Copied 2c_concatenat